# 17 · Tracing, Observability & Evals

Trace LLM apps (LangChain, OpenAI, Phoenix/Langfuse) and run evals on those traces.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfzalManihar/Agentic-AI-Learning/blob/main/17_tracing_observability_evals.ipynb)

**Covered in this notebook**

- Evaluation & Observability For LLM System
- Tracing & Observability for LLM Systems — Practical Demo
- Setup
- Part 0 — The problem: "the output was wrong"
- Span and trace — the core idea
- Debugging becomes inspection, not guessing
- Observability tells you *what happened*. Evaluation tells you *was it good*.
- Part 1 — LangSmith
- 1.1 A LangChain app is traced automatically
- 1.2 Non-LangChain code: use `@traceable`
- 1.3 Datasets and evals (a LangSmith specialty)
- LangSmith — advantages and disadvantages

> Part of [Agentic AI Learning](https://github.com/AfzalManihar/Agentic-AI-Learning). Add your own API keys in Colab Secrets (key icon) before running.

#11-10-2026
# Evaluation & Observability For LLM System

# Tracing & Observability for LLM Systems — Practical Demo

In normal software, a test **passes or fails**. In LLM systems, *"is this a good answer?"* is not binary:
an answer can be accurate but vague, fluent but hallucinated in one number, well formatted but badly reasoned,
or relevant but **not grounded in the retrieved evidence**. There is **no unit test for "did the agent reason well?"**

> **Analogy from your notes:** Project Manager -> Developer builds the website -> **Tester / QA (Quality Analyst)** checks the button, the login, and so on.
> For LLM apps, the "QA" is **observability + evaluation**.

| Discipline | Question it answers | What it does |
|---|---|---|
| **Observability** | *"What happened?"* | Records every prompt, retrieval, tool call and token |
| **Evaluation** | *"Was it good?"* | Scores outputs with metrics, LLM judges and humans |

You need **both**.

### The four tools from your notes (we use all of them)
| # | Tool | What it is | Needs an account? |
|---|---|---|---|
| 1 | **LangSmith** | LangChain's native platform. Set an environment variable and traces appear. Includes datasets and evals. | Yes (free) |
| 2 | **Langfuse** | Open source, framework agnostic, **self-hostable**. Good when data cannot leave your infrastructure. | Yes (free cloud) |
| 3 | **OpenTelemetry for LLMs** | Vendor-neutral standard (OpenLLMetry, OpenInference). **Instrument once, send to any backend.** | **No** |
| 4 | **Arize Phoenix** | Open source. Strong evals, RAGAS support, drift detection. Runs **inside Colab**. | **No** |

### One story for the whole notebook
**ShopEasy** has a support agent: it *retrieves* policy documents and asks an LLM to answer from them.
It has a hidden bug (a stale search index). We will find that bug with every tool and see what each one is best at.

**Setup you need:** a Groq key in Colab Secrets named `groq_api`.
**Optional keys** (a part is simply skipped if its key is missing): `langsmith_api`, `langfuse_public_key`, `langfuse_secret_key`.
Run the cells **top to bottom**.

### Notebook map
| Part | Topic | Extra keys? |
|---|---|---|
| 0 | The problem + a tracer built from scratch | no |
| 1 | LangSmith | optional |
| 2 | Langfuse | optional |
| 3 | OpenTelemetry for LLMs | no |
| 4 | Arize Phoenix | no |
| 5 | **Evaluation**: reference-based vs reference-free, LLM as judge, the four judge modes, the five judge biases | no (same Groq key) |
| 6 | **Judges inside** LangSmith, Langfuse and Phoenix (offline + online evaluation) | the same optional keys |
| 7 | **Humans**: calibrate the judge (you type once) | no |
| 8 | Which tool? Which evaluation? Checklist, takeaways, troubleshooting | no |

## Setup

In [5]:
# ===== Cell 3 | install =====
# Install everything once (takes 2-3 minutes).
# If Colab asks you to RESTART the session after this cell, click Restart and continue from the next cell (do not re-run this one).
!pip install -q groq openai langsmith langchain-groq langfuse opentelemetry-sdk opentelemetry-exporter-otlp-proto-http openinference-instrumentation-groq arize-phoenix

In [6]:
# ===== Cell 4 | setup =====
import os, re, json, time
import pandas as pd
from groq import Groq
from google.colab import userdata

def get_secret(name):
    """Read a Colab secret. Returns None if it does not exist, so optional parts can be skipped."""
    try:
        return userdata.get(name)
    except Exception:
        return None

GROQ_KEY = get_secret("groq_api")
assert GROQ_KEY, "Save your Groq key in Colab Secrets with the name groq_api and turn on Notebook access."

client = Groq(api_key=GROQ_KEY)
MODEL = "openai/gpt-oss-20b"          # one small, fast Groq model for everything

def llm_call(messages, max_tokens=800):
    """One REAL Groq call. Returns (text, usage). Waits and retries if Groq says 'rate limit'."""
    for _ in range(6):
        try:
            r = client.chat.completions.create(model=MODEL, messages=messages,
                                               max_tokens=max_tokens, temperature=0.2)
            usage = {"prompt_tokens": r.usage.prompt_tokens,
                     "completion_tokens": r.usage.completion_tokens}
            return (r.choices[0].message.content or "").strip(), usage
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e):
                time.sleep(10)                       # free tier: too many tokens per minute
            else:
                raise
    raise RuntimeError("Groq did not answer")

print(llm_call([{"role": "user", "content": "Say 'ready' in one word."}])[0])

ready


---
# Part 0 — The problem: "the output was wrong"
ShopEasy's agent retrieves policy documents and asks the LLM to answer from them.
A customer asks about returning a laptop. The next cell runs the agent with **no tracing**.

In [7]:
# ===== Cell 6 | the agent (no tracing) + hidden bug =====
# A small company knowledge base (the "documents" our agent can read)
DOCS = {
    "policy_v1_2022": "OLD POLICY (2022): Returns are accepted within 30 days of delivery for any item, including laptops.",
    "policy_v2_2025": "RETURN POLICY (2025): Returns are accepted within 7 days of delivery. Electronics such as laptops and phones must be unopened or defective.",
    "shipping":       "Shipping: standard delivery takes 3 to 5 business days. Express delivery takes 1 to 2 days.",
    "refund":         "Refunds: money goes back to the original payment method within 5 to 7 business days.",
}

# THE BUG: the search index is stale. It still holds the 2022 policy and the new 2025 policy was never added.
STALE_INDEX = ["policy_v1_2022", "shipping", "refund"]
FIXED_INDEX = ["policy_v2_2025", "shipping", "refund"]
INDEX = list(STALE_INDEX)

def use_index(which):
    """Switch between the stale (buggy) and the fixed index."""
    global INDEX
    INDEX = list(STALE_INDEX if which == "stale" else FIXED_INDEX)

STOP = {"the", "a", "an", "to", "of", "do", "i", "it", "have", "how", "many", "is", "for", "my", "and", "or", "within", "can", "bought", "long", "does", "take"}

def words(text):
    """Lower-case words, no stop words, plural 's' removed (so 'days' matches 'day')."""
    return {w.rstrip("s") for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP}

def retrieve(question, k=2):
    """Very simple retriever: rank documents by how many words they share with the question."""
    q = words(question)
    scored = [(len(q & words(DOCS[d])), d) for d in INDEX]
    scored.sort(key=lambda x: -x[0])
    return [{"id": d, "text": DOCS[d], "score": s} for s, d in scored[:k]]

SYSTEM_PROMPT = "You are a support agent for ShopEasy. Answer in 1-2 sentences using ONLY the context."

def build_messages(question, chunks):
    context = "\n".join("- " + c["text"] for c in chunks)
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}]

def answer(question):
    chunks = retrieve(question)
    return llm_call(build_messages(question, chunks))[0]

QUESTION = "I bought a laptop. How many days do I have to return it?"
print("ANSWER:", answer(QUESTION))

ANSWER: You have 30 days from the delivery date to return the laptop.


**What you should see:** the answer says **30 days**. That is **wrong** (the 2025 policy says 7 days).
All we know right now is *"the output was wrong"*. Why? We can only guess. That is the pain of debugging without tracing.

## Span and trace — the core idea
- A **span** is **one recorded operation**: an LLM call, a retrieval, a tool call. It stores start time, end time, input, output, model, token counts, latency and cost.
- Spans **nest into a trace**: an agent span contains retrieval and LLM-call spans. One request = **one tree**.

```
🤖 support_agent          <- the whole request (agent span)
   ├── 🔎 retrieve         <- which chunks came back?
   └── 🧠 llm_call         <- what exact context went in? what came out?
```

Before using any tool, we build a tiny tracer **from scratch**, so you see there is no magic.

In [8]:
# ===== Cell 8 | tracer from scratch =====
import time
from dataclasses import dataclass, field

# Illustrative prices in USD per 1 million tokens (input, output). Check Groq's pricing page for real numbers.
PRICE_PER_M = {"openai/gpt-oss-20b": (0.075, 0.30)}

@dataclass
class Span:
    """ONE recorded operation (an LLM call, a retrieval, a tool call, or a whole agent run)."""
    name: str
    kind: str                       # "agent", "retriever", "llm", "tool"
    input: object = None
    output: object = None
    model: str = None
    tokens_in: int = 0
    tokens_out: int = 0
    start: float = 0.0
    end: float = 0.0
    error: str = None
    children: list = field(default_factory=list)   # spans nest -> a tree

    @property
    def latency_ms(self):
        return (self.end - self.start) * 1000

    @property
    def cost_usd(self):
        p_in, p_out = PRICE_PER_M.get(self.model, (0, 0))
        return (self.tokens_in * p_in + self.tokens_out * p_out) / 1_000_000

_open_spans = []     # spans that are running right now (the top one is the current parent)
TRACES = []          # finished traces (each trace = one root span with all its children)

class span:
    """Use:  with span("name", "kind", input=...) as s:  ...do work...  s.output = ..."""
    def __init__(self, name, kind, input=None, model=None):
        self.s = Span(name=name, kind=kind, input=input, model=model)

    def __enter__(self):
        self.s.start = time.perf_counter()
        if _open_spans:
            _open_spans[-1].children.append(self.s)      # attach to the parent span
        _open_spans.append(self.s)
        return self.s

    def __exit__(self, exc_type, exc, tb):
        self.s.end = time.perf_counter()
        if exc:
            self.s.error = repr(exc)                     # failures are recorded too
        _open_spans.pop()
        if not _open_spans:
            TRACES.append(self.s)                        # root span closed -> trace is complete
        return False

def print_trace(s, depth=0):
    """Print the trace as a tree."""
    icon = {"agent": "🤖", "retriever": "🔎", "llm": "🧠", "tool": "🔧"}.get(s.kind, "•")
    extra = f" | tokens {s.tokens_in}->{s.tokens_out} | ${s.cost_usd:.6f}" if s.kind == "llm" else ""
    err = f" | ERROR {s.error}" if s.error else ""
    print("   " * depth + f"{icon} {s.name} [{s.kind}] {s.latency_ms:.0f} ms{extra}{err}")
    for c in s.children:
        print_trace(c, depth + 1)

def traced_answer(question):
    """Same logic as answer(), but every step is recorded as a span."""
    with span("support_agent", "agent", input=question) as agent:
        with span("retrieve", "retriever", input=question) as r:
            chunks = retrieve(question)
            r.output = chunks                              # exactly which chunks came back, in order
        messages = build_messages(question, chunks)
        with span("llm_call", "llm", input=messages, model=MODEL) as s:
            text, usage = llm_call(messages)
            s.output = text
            s.tokens_in, s.tokens_out = usage["prompt_tokens"], usage["completion_tokens"]
        agent.output = text
    return text

print("Tracer ready")

Tracer ready


In [9]:
# ===== Cell 9 | run with tracing =====
TRACES.clear()
print("ANSWER:", traced_answer(QUESTION))
STALE_TRACE = TRACES[-1]            # keep this trace, we will compare it later

print("\nTHE TRACE (one request = one tree):")
print_trace(STALE_TRACE)

ANSWER: You have 30 days from delivery to return the laptop.

THE TRACE (one request = one tree):
🤖 support_agent [agent] 468 ms
   🔎 retrieve [retriever] 0 ms
   🧠 llm_call [llm] 468 ms | tokens 161->109 | $0.000045


**What you should see:** the answer, then a tree: `support_agent` contains `retrieve` and `llm_call`.
Every span shows its latency, and the LLM span shows tokens (for example `89->14`) and cost.

## Debugging becomes inspection, not guessing
Without tracing we only knew *"the output was wrong"*. With the trace we can open each span and check the
**query**, the **chunks returned** and their order, the **exact context given to the LLM**, and the **output**.

In [10]:
# ===== Cell 11 | inspect the trace =====
retriever_span = STALE_TRACE.children[0]
llm_span = STALE_TRACE.children[1]

print("1) QUERY sent to the retriever  :", retriever_span.input)
print("2) CHUNKS the retriever returned:")
for c in retriever_span.output:
    print(f"     score {c['score']} | {c['id']}")
print("3) CONTEXT given to the LLM     :")
print(llm_span.input[1]["content"])
print("4) FINAL OUTPUT                 :", llm_span.output)

1) QUERY sent to the retriever  : I bought a laptop. How many days do I have to return it?
2) CHUNKS the retriever returned:
     score 3 | policy_v1_2022
     score 1 | shipping
3) CONTEXT given to the LLM     :
Context:
- OLD POLICY (2022): Returns are accepted within 30 days of delivery for any item, including laptops.
- Shipping: standard delivery takes 3 to 5 business days. Express delivery takes 1 to 2 days.

Question: I bought a laptop. How many days do I have to return it?
4) FINAL OUTPUT                 : You have 30 days from delivery to return the laptop.


**Root cause found:** the chunks are `policy_v1_2022` (OLD POLICY) and `shipping`. The new `policy_v2_2025` **never reached the LLM**.
The LLM was not wrong; the **retriever** was. Now we fix the index and trace again.

In [11]:
# ===== Cell 13 | fix and re-trace =====
use_index("fixed")                      # THE FIX: the outdated 2022 policy is removed and the 2025 policy is indexed

TRACES.clear()
print("ANSWER:", traced_answer(QUESTION))
FIXED_TRACE = TRACES[-1]

print()
print_trace(FIXED_TRACE)
print("\nChunks now retrieved:", [c["id"] for c in FIXED_TRACE.children[0].output])

ANSWER: You have 7 days from the delivery date to return the laptop, provided it is unopened or defective.

🤖 support_agent [agent] 232 ms
   🔎 retrieve [retriever] 0 ms
   🧠 llm_call [llm] 232 ms | tokens 167->104 | $0.000044

Chunks now retrieved: ['policy_v2_2025', 'shipping']


**What you should see:** the answer now says **7 days** and the chunks are `['policy_v2_2025', 'shipping']`.

## Observability tells you *what happened*. Evaluation tells you *was it good*.
Let us score **both traces**. We use two LLM judges:
- **grounded_in_context** — is the answer supported by the context the LLM saw?
- **correct** — does the answer match the real policy?

Watch the stale trace: it should be **grounded but not correct**. That combination points to a **retrieval** problem, not an LLM problem.

In [12]:
# ===== Cell 15 | evaluate the two traces =====
TRUTH = "Laptops can be returned within 7 days of delivery, and they must be unopened or defective."

def judge(prompt):
    """LLM-as-judge: one real call that must reply YES or NO."""
    text = llm_call([{"role": "system", "content": "You are a strict evaluator. Reply with exactly one word: YES or NO."},
                     {"role": "user", "content": prompt}])[0]
    return "YES" if text.strip().upper().startswith("YES") else "NO"

def evaluate_trace(trace):
    question, answer_text = trace.input, trace.output
    context = trace.children[1].input[1]["content"]
    return {
        "grounded_in_context": judge(f"{context}\n\nAnswer: {answer_text}\n\nIs every claim in the answer supported by the context above?"),
        "correct": judge(f"Correct fact: {TRUTH}\n\nQuestion: {question}\nAnswer: {answer_text}\n\nDoes the answer agree with the correct fact?"),
    }

rows = []
for name, tr in [("stale index (buggy)", STALE_TRACE), ("fixed index", FIXED_TRACE)]:
    rows.append({"run": name, "answer": tr.output[:70], **evaluate_trace(tr)})
pd.DataFrame(rows)

,run,answer,grounded_in_context,correct
0,stale index (buggy),You have 30 days from delivery to return the l...,YES,NO
1,fixed index,You have 7 days from the delivery date to retu...,YES,YES


**What you should see:** stale run -> `grounded_in_context = YES`, `correct = NO`. Fixed run -> both `YES`.
(LLM judges can vary a little between runs; that is why serious teams also add human review.)

Our hand-made tracer works, but real projects use a **tool** so traces are stored, searchable, shared and linked to evals.
Now the four tools, one by one, on the **same bug**.

---
# Part 1 — LangSmith
**What it is:** LangChain's native platform. **Set an environment variable and traces appear.** It also has **datasets and evals**.

**Best when:** you build with LangChain / LangGraph and want the fastest path from "no tracing" to "full traces + evals".

**Get a free key:** smith.langchain.com -> Settings -> API Keys. Save it in Colab Secrets as `langsmith_api`.
If the key is missing, this part still runs, just without sending traces.

In [13]:
# ===== Cell 18 | LangSmith setup (only environment variables) =====
LS_KEY = get_secret("langsmith_api")
HAS_LANGSMITH = bool(LS_KEY)

if HAS_LANGSMITH:
    os.environ["LANGSMITH_TRACING"] = "true"          # <- this one line turns tracing on
    os.environ["LANGSMITH_API_KEY"] = LS_KEY
    os.environ["LANGSMITH_PROJECT"] = "shopeasy-demo"  # traces are grouped by project
    # If your account is in the EU region, also set: os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
    print("LangSmith tracing is ON -> project 'shopeasy-demo'")
else:
    print("No langsmith_api secret found. The code below still runs, but nothing is sent to LangSmith.")

No langsmith_api secret found. The code below still runs, but nothing is sent to LangSmith.


### 1.1 A LangChain app is traced automatically
We rebuild the agent as a **LangChain chain**: retrieve -> prompt -> LLM -> parser.
We write **no tracing code at all**. The environment variable is enough. We run it on the **stale** index first.

In [14]:
# ===== Cell 20 | LangChain chain (auto traced) =====
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

lc_llm = ChatGroq(model=MODEL, temperature=0.2, max_tokens=800, api_key=GROQ_KEY)

def retrieve_context(question):
    """The retrieval step. LangSmith shows it as its own span."""
    return "\n".join("- " + c["text"] for c in retrieve(question))

lc_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

chain = ({"context": RunnableLambda(retrieve_context), "question": RunnablePassthrough()}
         | lc_prompt | lc_llm | StrOutputParser())

use_index("stale")
print("STALE index ->", chain.invoke(QUESTION))
use_index("fixed")
print("FIXED index ->", chain.invoke(QUESTION))

STALE index -> You have 30 days from the delivery date to return the laptop.
FIXED index -> You have 7 days from the delivery date to return the laptop, provided it is unopened or defective.


**What you should see:** two answers: **30 days** (stale) and **7 days** (fixed).

**In LangSmith:** open smith.langchain.com -> *Tracing* -> project `shopeasy-demo`. You will see two traces. Open the first one:
the tree shows `RunnableSequence` -> `retrieve_context` -> `ChatPromptTemplate` -> `ChatGroq` -> `StrOutputParser`,
with the exact context, tokens, latency and cost on each node. The stale trace shows the 2022 policy in the context.

### 1.2 Non-LangChain code: use `@traceable`
Not every app uses LangChain. The `@traceable` decorator gives the same nested spans for **plain Python** (our own `retrieve` and Groq call).

In [15]:
# ===== Cell 22 | @traceable on plain Python =====
from langsmith import traceable
from langsmith.run_helpers import get_current_run_tree

@traceable(run_type="retriever", name="retrieve")
def ls_retrieve(question):
    return retrieve(question)

@traceable(run_type="llm", name="groq_call")
def ls_llm(messages):
    text, usage = llm_call(messages)
    run = get_current_run_tree()                      # the span we are inside (None if tracing is off)
    if run:
        run.metadata.update({"ls_provider": "groq", "ls_model_name": MODEL})
        run.set(usage_metadata={"input_tokens": usage["prompt_tokens"],
                                "output_tokens": usage["completion_tokens"],
                                "total_tokens": usage["prompt_tokens"] + usage["completion_tokens"]})
    return text

@traceable(run_type="chain", name="support_agent")
def ls_agent(question):
    chunks = ls_retrieve(question)
    return ls_llm(build_messages(question, chunks))

use_index("stale")
print("Plain-Python agent (stale index):", ls_agent(QUESTION))

Plain-Python agent (stale index): You have 30 days from delivery to return the laptop.


In [16]:
# ===== Cell 23 | read the trace back with the API =====
if HAS_LANGSMITH:
    from langsmith import Client
    ls = Client()
    ls.flush()                                         # make sure all traces are uploaded
    time.sleep(3)
    runs = list(ls.list_runs(project_name="shopeasy-demo", is_root=True, limit=3))
    for r in runs:
        secs = (r.end_time - r.start_time).total_seconds() if r.end_time else None
        print(f"{r.name:<18} status={r.status:<8} tokens={r.total_tokens}  latency={secs and round(secs, 2)}s")
else:
    print("Skipped (no LangSmith key).")

Skipped (no LangSmith key).


**What you should see:** the latest root runs (`support_agent`, `RunnableSequence`) with status, tokens and latency.
You can read traces back **from code**, not only from the website.

### 1.3 Datasets and evals (a LangSmith specialty)
A **dataset** is a list of test questions with expected answers. An **experiment** runs your agent on every question and scores it.
We compare the **stale** and the **fixed** index. This is a regression test for an LLM app.

In [17]:
# ===== Cell 25 | LangSmith dataset + evals =====
DATASET = "shopeasy-support-qa"

EXAMPLES = [
    {"question": "I bought a laptop. How many days do I have to return it?", "expected": "7 days"},
    {"question": "How long do refunds take?",                                   "expected": "5 to 7"},
    {"question": "How long does express delivery take?",                        "expected": "1 to 2"},
]

def target(inputs):
    """The function being tested: question in, answer out."""
    return {"answer": ls_agent(inputs["question"])}

SCORES = []    # we also keep the scores here, so we can print a summary table

def contains_expected(outputs, reference_outputs):
    """A simple evaluator: does the answer contain the expected fact?"""
    score = int(reference_outputs["expected"] in outputs["answer"])
    SCORES.append(score)
    return {"key": "contains_expected", "score": score}

if HAS_LANGSMITH:
    from langsmith import evaluate
    if not ls.has_dataset(dataset_name=DATASET):
        ds = ls.create_dataset(DATASET, description="ShopEasy support questions")
        ls.create_examples(dataset_id=ds.id,
                           examples=[{"inputs": {"question": e["question"]}, "outputs": {"expected": e["expected"]}} for e in EXAMPLES])

    summary = []
    for which in ["stale", "fixed"]:
        use_index(which)
        SCORES.clear()
        evaluate(target, data=DATASET, evaluators=[contains_expected],
                 experiment_prefix=f"{which}-index", max_concurrency=1)
        summary.append({"experiment": f"{which}-index", "passed": sum(SCORES), "of": len(SCORES)})
    display(pd.DataFrame(summary))
else:
    print("Skipped (no LangSmith key).")

Skipped (no LangSmith key).


**What you should see:** `stale-index` passes **2 of 3** (the laptop question fails) and `fixed-index` passes **3 of 3**.
In LangSmith -> *Datasets & Experiments* you can open both experiments and compare them side by side.

### LangSmith — advantages and disadvantages
| Advantages | Disadvantages |
|---|---|
| **One environment variable** traces a LangChain / LangGraph app | Best experience is inside the LangChain ecosystem |
| Datasets, experiments and evals in the same place | Hosted SaaS: your prompts and data go to LangSmith (self-host needs an enterprise plan) |
| `@traceable` works for plain Python too | Free tier has limits on trace volume |

---
# Part 2 — Langfuse
**What it is:** open source, **framework agnostic**, and **self-hostable**. Good when **data cannot leave your infrastructure**.

**Best when:** privacy rules (banking, health, government) say prompts must stay on your servers, or you use many frameworks.

**Get a free key:** cloud.langfuse.com -> create a project -> Settings -> API Keys. Save in Colab Secrets:
`langfuse_public_key`, `langfuse_secret_key` (and optionally `langfuse_host`, default `https://cloud.langfuse.com`; US region is `https://us.cloud.langfuse.com`).

In [18]:
# ===== Cell 28 | Langfuse setup =====
LF_PUBLIC = get_secret("langfuse_public_key")
LF_SECRET = get_secret("langfuse_secret_key")
LF_HOST = get_secret("langfuse_host") or "https://cloud.langfuse.com"
HAS_LANGFUSE = bool(LF_PUBLIC and LF_SECRET)

if HAS_LANGFUSE:
    os.environ["LANGFUSE_PUBLIC_KEY"] = LF_PUBLIC
    os.environ["LANGFUSE_SECRET_KEY"] = LF_SECRET
    os.environ["LANGFUSE_HOST"] = LF_HOST          # <- for a self-hosted server only THIS line changes
else:
    os.environ["LANGFUSE_TRACING_ENABLED"] = "false"   # no keys: the decorators below simply do nothing

from langfuse import get_client, observe, propagate_attributes
langfuse = get_client()

if HAS_LANGFUSE:
    try:
        print("Langfuse connection OK:", langfuse.auth_check())
    except Exception as e:
        print("Langfuse auth check failed. Check your keys and host:", e)
else:
    print("No Langfuse keys found. The code below still runs, but nothing is sent to Langfuse.")

No Langfuse keys found. The code below still runs, but nothing is sent to Langfuse.


### 2.1 Drop-in OpenAI client + `@observe`
Groq speaks the OpenAI API. Langfuse ships a **drop-in OpenAI client**: change one import line and every LLM call is traced automatically
(model, input, output, tokens, latency). `@observe` adds the parent spans (agent, retriever).

In [19]:
%pip install -q --upgrade --force-reinstall openai langfuse

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mem0ai 1.0.11 requires protobuf<7.0.0,>=5.29.6, but you have protobuf 7.36.2 which is incompatible.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
ibis-framework 9.5.0 requires sqlglot<25.21,>=23.4, but you have sqlglot 30.21.0 which is incompatible.


In [20]:
# ===== Cell 30 | Langfuse traced agent =====
from langfuse.openai import OpenAI as LangfuseOpenAI    # <- the only change: import the Langfuse version

lf_client = LangfuseOpenAI(api_key=GROQ_KEY, base_url="https://api.groq.com/openai/v1")

@observe(as_type="retriever", name="retrieve")
def lf_retrieve(question):
    return retrieve(question)

@observe(as_type="agent", name="support_agent")
def lf_agent(question):
    chunks = lf_retrieve(question)
    resp = lf_client.chat.completions.create(          # automatically recorded as an LLM "generation"
        model=MODEL, messages=build_messages(question, chunks), max_tokens=800, temperature=0.2, name="groq_call")
    return resp.choices[0].message.content.strip()

@observe(name="support_request")
def lf_request(question, expected):
    """One full request + an evaluation score attached to the same trace."""
    answer_text = lf_agent(question)
    langfuse.score_current_trace(name="contains_expected", value=int(expected in answer_text),
                                 comment=f"expected '{expected}'")
    return answer_text, langfuse.get_current_trace_id()

use_index("stale")
with propagate_attributes(user_id="riya", session_id="chat-001", tags=["support", "stale-index"]):
    text, trace_id = lf_request(QUESTION, "7 days")
print("ANSWER:", text)

langfuse.flush()                                       # send the traces now
if HAS_LANGFUSE:
    try:
        print("Open this trace:", langfuse.get_trace_url(trace_id=trace_id))
    except Exception as e:
        print("Trace sent. (Could not build the link:", e, ") Open your Langfuse project -> Tracing.")

ANSWER: You have 30 days from the delivery date to return the laptop.


**What you should see:** the answer (**30 days** on the stale index) and, if you added keys, a **link to the trace**.

**In Langfuse:** open the link. You will see `support_request` -> `support_agent` -> `retrieve` and `groq_call`,
the exact input and output, tokens and latency, plus the score **`contains_expected = 0`** attached to the trace.
Also note the **user** (`riya`), the **session** (`chat-001`) and the **tags**.

### 2.2 Users and sessions (a Langfuse specialty)
In production you need *"which user had the bad answer?"* and *"how much does each user cost?"*.
We simulate two users with two sessions.

In [21]:
# ===== Cell 32 | users and sessions =====
use_index("fixed")
traffic = [("riya", "chat-001", "I bought a laptop. How many days do I have to return it?", "7 days"),
           ("riya", "chat-001", "How long do refunds take?", "5 to 7"),
           ("arjun", "chat-002", "How long does express delivery take?", "1 to 2")]

rows = []
for user_id, session_id, question, expected in traffic:
    with propagate_attributes(user_id=user_id, session_id=session_id, tags=["support", "fixed-index"]):
        text, tid = lf_request(question, expected)
    rows.append({"user": user_id, "session": session_id, "question": question[:40], "correct": expected in text})

langfuse.flush()
pd.DataFrame(rows)

,user,session,question,correct
0,riya,chat-001,I bought a laptop. How many days do I ha,True
1,riya,chat-001,How long do refunds take?,True
2,arjun,chat-002,How long does express delivery take?,True


**In Langfuse:** the *Users* tab shows `riya` and `arjun` with their traces and token cost, and the *Sessions* tab groups `chat-001` as one conversation.

### 2.3 Why Langfuse for "data cannot leave our infrastructure"
Langfuse is open source, so you can run **your own server**. The code does not change. Only the host does:

```bash
git clone https://github.com/langfuse/langfuse.git && cd langfuse
docker compose up -d                  # Langfuse now runs on your own machine / VPC
```
```python
os.environ["LANGFUSE_HOST"] = "https://langfuse.mycompany.internal"   # same code, your server
```
(Colab cannot run Docker, so we use the free cloud here. The self-hosting steps above are not executed in this notebook.)

### Langfuse — advantages and disadvantages
| Advantages | Disadvantages |
|---|---|
| **Self-hostable**: prompts and user data stay in your infrastructure | Self-hosting means you run and maintain it (database, upgrades) |
| Framework agnostic; drop-in OpenAI client and `@observe` | Fewer built-in eval tools than Phoenix / LangSmith |
| Users, sessions, tags, scores and cost in one place | Cloud free tier has volume limits |

---
# Part 3 — OpenTelemetry for LLMs
**What it is:** a **vendor-neutral standard**. LLM-specific flavours are **OpenLLMetry** and **OpenInference**.
**Instrument once, send to any backend.** (Your notes say: *still maturing*.)

**Best when:** your company already uses OpenTelemetry, you do not want to be locked to one vendor, or your app mixes many languages.

**No account needed.** We use **OpenInference**: its Groq instrumentor patches the Groq library, so
**every Groq call becomes a span automatically** (model, messages, tokens). We only add the agent and retriever spans ourselves.

In [22]:
# ===== Cell 35 | OpenTelemetry setup (instrument once) =====
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from openinference.instrumentation.groq import GroqInstrumentor

PROJECT = "shopeasy-demo"
provider = TracerProvider(resource=Resource.create({"service.name": "shopeasy",
                                                    "openinference.project.name": PROJECT}))   # Phoenix uses this as project name

memory_exporter = InMemorySpanExporter()                          # backend #1: keep spans in Python so we can print them
provider.add_span_processor(SimpleSpanProcessor(memory_exporter))

GroqInstrumentor().instrument(tracer_provider=provider)           # <- instrument ONCE: every Groq call is now a span
tracer = provider.get_tracer("shopeasy")
print("OpenTelemetry ready. Groq calls are now traced automatically.")

OpenTelemetry ready. Groq calls are now traced automatically.


In [23]:
# ===== Cell 36 | agent with OpenTelemetry spans =====
def otel_answer(question):
    with tracer.start_as_current_span("support_agent") as agent:
        agent.set_attribute("openinference.span.kind", "AGENT")
        agent.set_attribute("input.value", question)

        with tracer.start_as_current_span("retrieve") as r:
            r.set_attribute("openinference.span.kind", "RETRIEVER")
            r.set_attribute("input.value", question)
            chunks = retrieve(question)
            r.set_attribute("output.value", json.dumps(chunks))

        # NOTE: llm_call() is our normal function. We add NO tracing code here.
        # The Groq instrumentor creates the LLM span (model, tokens, messages) by itself.
        text, _ = llm_call(build_messages(question, chunks))

        agent.set_attribute("output.value", text)
    return text

use_index("stale")
memory_exporter.clear()
print("ANSWER:", otel_answer(QUESTION))

ANSWER: You have 30 days from delivery to return the laptop.


In [24]:
# ===== Cell 37 | print the OpenTelemetry trace as a tree =====
def print_otel_tree(exporter):
    spans = exporter.get_finished_spans()
    by_parent = {}
    for s in spans:
        by_parent.setdefault(s.parent.span_id if s.parent else None, []).append(s)

    def show(s, depth):
        a = dict(s.attributes)
        ms = (s.end_time - s.start_time) / 1e6
        kind = a.get("openinference.span.kind", "")
        extra = ""
        if kind == "LLM":
            extra = f" | model={a.get('llm.model_name')} | tokens {a.get('llm.token_count.prompt')}->{a.get('llm.token_count.completion')}"
        print("   " * depth + f"{s.name} [{kind}] {ms:.0f} ms{extra}")
        for child in sorted(by_parent.get(s.context.span_id, []), key=lambda x: x.start_time):
            show(child, depth + 1)

    for root in by_parent.get(None, []):
        show(root, 0)

print_otel_tree(memory_exporter)

support_agent [AGENT] 481 ms
   retrieve [RETRIEVER] 0 ms
   Completions [LLM] 480 ms | model=openai/gpt-oss-20b | tokens 161->137


**What you should see:** a tree like `support_agent [AGENT]` with `retrieve [RETRIEVER]` and `Completions [LLM]`.
The LLM span has the **model name and token counts**, and we wrote **no tracing code** around the Groq call.

### Instrument once, send to any backend
Right now our spans go to a Python list (`InMemorySpanExporter`). In Part 4 we will add a **second exporter** that sends the same spans to **Phoenix**,
without changing a single line of the agent code. Tomorrow it could be Datadog, Grafana, Jaeger or Langfuse. That is the point of the standard.

### OpenLLMetry vs OpenInference
Both are OpenTelemetry-based "dictionaries" for LLM attributes. We used **OpenInference** (made by Arize, so Phoenix reads it natively).
**OpenLLMetry** (by Traceloop) is the other one: you install `traceloop-sdk` and call `Traceloop.init()`. (We did not run OpenLLMetry in this notebook.)

### OpenTelemetry — advantages and disadvantages
| Advantages | Disadvantages |
|---|---|
| **No vendor lock-in**: change the backend, keep the code | You must choose and run a backend yourself |
| Works across languages and with your existing OTel setup | LLM conventions are **still maturing** (names and attributes change) |
| Auto-instrumentation for many LLM libraries | No evals, datasets or prompt tools of its own |

---
# Part 4 — Arize Phoenix
**What it is:** open source, strong **evals**, **RAGAS** support and **drift detection**. It runs **inside Colab**, with **no account**.

**Best when:** you debug a **RAG** system in a notebook, want traces and evals in one place, and want it free and local.

We start the Phoenix app, then plug it into the OpenTelemetry setup from Part 3 as a **second backend**.

In [25]:
# ===== Cell 40 | start Phoenix =====
import phoenix as px

session = px.launch_app()                    # starts the Phoenix server inside this Colab machine
print("Phoenix UI:", session.url)

/usr/lib/python3.13/contextlib.py:148: SAWarning: Skipped unsupported reflection of expression-based index ix_cumulative_llm_token_count_total
  next(self.gen)
/usr/lib/python3.13/contextlib.py:148: SAWarning: Skipped unsupported reflection of expression-based index ix_latency
  next(self.gen)
/usr/lib/python3.13/contextlib.py:148: SAWarning: Skipped unsupported reflection of expression-based index ix_spans_session_id
  next(self.gen)
/usr/local/lib/python3.13/dist-packages/pydantic/json_schema.py:2666: PydanticJsonSchemaWarning: Default value <phoenix.db.types.db_helper_types.Undefined object at 0x7e16315fb0e0> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
<sys>:0: SAWarning: Column-expression-level unary distinct() should not be used outside of an aggregate function. For general 'SELECT DISTINCT' supportuse select().distinct().


🌍 To view the Phoenix app in your browser, visit https://70ftemnktr61-496ff2e9c6d22116-6006-colab.googleusercontent.com/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
Phoenix UI: https://70ftemnktr61-496ff2e9c6d22116-6006-colab.googleusercontent.com/


Click the link above to open the Phoenix UI (it is empty for now). Next we send the spans from Part 3 to it.

In [26]:
# ===== Cell 42 | send the same spans to Phoenix =====
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter

# backend #2: add ONE line. The agent code stays exactly the same.
provider.add_span_processor(SimpleSpanProcessor(OTLPSpanExporter(endpoint="http://localhost:6006/v1/traces")))

for which in ["stale", "fixed"]:
    use_index(which)
    print(f"{which.upper():<6} index ->", otel_answer(QUESTION))

provider.force_flush()
time.sleep(3)
print("\nSpans sent. Open the Phoenix UI link -> project 'shopeasy-demo'.")

STALE  index -> You have 30 days from the delivery date to return the laptop.
FIXED  index -> You have 7 days from the delivery date to return the laptop, provided it is unopened or defective.

Spans sent. Open the Phoenix UI link -> project 'shopeasy-demo'.


**In the Phoenix UI:** open project `shopeasy-demo`. You see two traces. Click one: the tree shows `support_agent`, `retrieve` and the Groq LLM span,
with input, output, tokens and latency. The retriever span's output holds the chunks, so you can spot the stale 2022 policy in one click.

### Traces as a DataFrame
Phoenix also lets you read your spans **from code**. This is great for notebooks and for running evals.

In [27]:
# ===== Cell 44 | read spans from Phoenix =====
from phoenix.client import Client as PhoenixClient

px_client = PhoenixClient(base_url="http://localhost:6006")
spans = px_client.spans.get_spans_dataframe(project_identifier=PROJECT)

view = spans[["name", "span_kind", "attributes.llm.token_count.prompt", "attributes.llm.token_count.completion"]].copy()
view["latency_ms"] = ((spans["end_time"] - spans["start_time"]).dt.total_seconds() * 1000).round()
view.rename(columns={"attributes.llm.token_count.prompt": "tokens_in",
                     "attributes.llm.token_count.completion": "tokens_out"})

,name,span_kind,tokens_in,tokens_out,latency_ms
context.span_id,,,,,
0905a50cefc0a2dd,Completions,LLM,167.0,98.0,174.0
1542ee7da6339ea5,retrieve,RETRIEVER,NaN,NaN,0.0
2b54a02443fad965,support_agent,AGENT,NaN,NaN,198.0
03fcbb5a5a161dca,Completions,LLM,161.0,93.0,260.0
c3cf4ebb42360917,retrieve,RETRIEVER,NaN,NaN,0.0
be6423c42648f658,support_agent,AGENT,NaN,NaN,289.0


### Evals on your traces
Phoenix is strong at **evaluating the traced data**. We take every `support_agent` span from Phoenix, read the retrieved context from its `retrieve` span,
score the answer with an LLM judge, and **write the scores back to Phoenix** so they appear next to the traces.

In [28]:
# ===== Cell 46 | evals on Phoenix traces =====
agents = spans[spans["name"] == "support_agent"]

records = []
for span_id, row in agents.iterrows():
    trace_id = row["context.trace_id"]
    retriever = spans[(spans["context.trace_id"] == trace_id) & (spans["name"] == "retrieve")].iloc[0]
    chunks = json.loads(retriever["attributes.output.value"])
    context = "\n".join("- " + c["text"] for c in chunks)
    question, answer_text = row["attributes.input.value"], row["attributes.output.value"]

    is_correct = judge(f"Correct fact: {TRUTH}\n\nQuestion: {question}\nAnswer: {answer_text}\n\nDoes the answer agree with the correct fact?")
    records.append({"context.span_id": span_id,
                    "label": "correct" if is_correct == "YES" else "incorrect",
                    "score": 1.0 if is_correct == "YES" else 0.0,
                    "explanation": "retrieved: " + ", ".join(c["id"] for c in chunks)})

eval_df = pd.DataFrame(records).set_index("context.span_id")
px_client.spans.log_span_annotations_dataframe(dataframe=eval_df, annotation_name="correctness", annotator_kind="LLM", sync=True)
eval_df

,label,score,explanation
context.span_id,,,
2b54a02443fad965,correct,1.0,"retrieved: policy_v2_2025, shipping"
be6423c42648f658,incorrect,0.0,"retrieved: policy_v1_2022, shipping"


**What you should see:** two rows: one `incorrect` (retrieved `policy_v1_2022`) and one `correct` (retrieved `policy_v2_2025`).
In the Phoenix UI each trace now shows a **correctness** annotation, so you can filter for bad answers and open their traces.

### More Phoenix features from your notes (not run here)
- **Native RAGAS support:** metrics such as faithfulness and context precision/recall can be computed over traced RAG data (the `ragas` library plugs into Phoenix).
- **Drift detection:** Phoenix can compare embeddings of today's queries with a baseline to spot topics your system was never tested on.
- **`phoenix.evals`:** a library of ready-made LLM-judge evaluators (hallucination, relevance, and so on). We wrote our own judge above to keep the dependency list small.

### Phoenix — advantages and disadvantages
| Advantages | Disadvantages |
|---|---|
| **Free, local, no account**: runs inside the notebook | Local server stops when the Colab session ends (data is not kept) |
| Strong evals and RAG debugging, built on OpenTelemetry / OpenInference | Less focused on users, sessions and team dashboards than Langfuse |
| Spans come back as DataFrames for analysis | Heavier install than the others |

---
# Part 5 — Evaluation: *"Was it good?"* (Step 2 of your notes)

Parts 1–4 gave us **observability** ("what happened?"). Now the second discipline: **evaluation** ("was it good?").

Our quick judge in Cell 15 was a start, but it took three shortcuts. In this Part we fix all three:

| Shortcut in Cell 15 | What we do properly now |
|---|---|
| The **same model** wrote the answer *and* judged it | A **weak** model writes. A **strong model from a different family** judges. (Your diagram: *Agent -> answer -> Strong LLM check*.) |
| The judge only said YES / NO | Structured **scores + a reason** (JSON), so you can read *why* |
| We never asked *"can I trust the judge?"* | We test the judge for its **five known biases** (Part 5.4) and compare it with a **human** (Part 7) |

### Reference-based vs reference-free (from your notes)
| | **Reference-based** | **Reference-free** |
|---|---|---|
| How it works | Compare the output to a **ground-truth answer** | Judge the output **on its own properties**: consistent? faithful to the context? on topic? |
| Works well for | **Extractive** tasks with one right answer | **Open-ended** generation |
| Problem | Many answers are correct, and string overlap **punishes valid paraphrases** | Harder to trust (that is why we test the judge) |
| Scales? | No: someone must write ground truth for every case | **Yes**: nobody writes ground truth for every case |

> **Your notes' conclusion:** for evaluation, the better *scalable* technique is **reference-free**, and reference-free means **LLM as judge**.

### The setup for this whole Part
```
 question ──►  🤖 Agent (WEAK LLM)  ──►  answer  ──►  🧑‍⚖️ Judge (STRONG LLM)  ──►  score + reason
                     ▲                                        ▲
                 context                         criteria YOU specify
```
**No new accounts or keys.** It uses the same `groq_api` key. This Part makes roughly 100 real Groq calls. On the free tier it can take **5-10 minutes** and you may see
`(rate limit - waiting ...)` messages. That is normal: our helper waits and retries automatically. Each sub-section runs on its own, so you can stop after any of them.

## Setup for evaluation: one weak generator, strong judges, safe helpers
We need three Groq models, chosen on purpose:

| Role | Model | Family | Why |
|---|---|---|---|
| **Weak generator** (writes answers) | `llama-3.1-8b-instant` | Meta | Small and fast, so it makes realistic mistakes |
| **Strong judge #1** | `openai/gpt-oss-120b` | OpenAI | Judges the Meta model's answers (**different family**) |
| **Strong judge #2** | `llama-3.3-70b-versatile` | Meta | Judges answers written by OpenAI-family models (our Part 1-4 agent used `gpt-oss-20b`) |

The helper `chat()` makes one **real** Groq call for *any* model, waits when Groq says "rate limit", and counts tokens so we can see **what evaluation costs**.
The cell first asks Groq which models exist right now, so if a model is ever retired you get a clear message instead of a confusing error.

In [32]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

for model in client.models.list().data:
    print(model.id)

openai/gpt-oss-safeguard-20b
allam-2-7b
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-20b
canopylabs/orpheus-arabic-saudi
whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-120b
whisper-large-v3
qwen/qwen3.8-27b
canopylabs/orpheus-v1-english


In [33]:

# ===== Cell 50 | Fixed evaluation setup =====

import random
import re
import json
import time
import pandas as pd
from collections import Counter, defaultdict

from groq import Groq
from google.colab import userdata

# Load your existing Colab Secret
GROQ_API_KEY = userdata.get("groq_api")
client = Groq(api_key=GROQ_API_KEY)

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 220)


# 1. Model family detection
def family(model_id):
    if model_id.startswith("openai/"):
        return "openai"
    if "llama" in model_id:
        return "meta"
    if "qwen" in model_id:
        return "qwen"
    return model_id.split("/")[0]


# 2. Fetch currently available models
try:
    AVAILABLE = {m.id for m in client.models.list().data}
    print(f"Available models found: {len(AVAILABLE)}")
except Exception as e:
    raise RuntimeError(
        f"Could not fetch Groq models. Check API key/access. Details: {e}"
    )


# 3. Select models from your available list
def pick(candidates, role):
    for model in candidates:
        if model in AVAILABLE:
            return model

    raise RuntimeError(
        f"No available model found for {role}. "
        f"Candidates: {candidates}"
    )


WEAK = pick(
    ["openai/gpt-oss-20b"],
    "weak generator"
)

JUDGE_OPENAI = pick(
    ["openai/gpt-oss-120b"],
    "strong OpenAI-family judge"
)

JUDGE_META = pick(
    ["qwen/qwen3.8-27b"],
    "second judge"
)


def pick_judge_for(generator_model):
    """Choose a judge from a different model family."""
    if family(generator_model) == family(JUDGE_OPENAI):
        return JUDGE_META
    return JUDGE_OPENAI


# 4. Track API calls and token usage
USAGE = defaultdict(
    lambda: {"calls": 0, "in": 0, "out": 0}
)


def _count(model, usage):
    if usage:
        USAGE[model]["calls"] += 1
        USAGE[model]["in"] += usage.prompt_tokens
        USAGE[model]["out"] += usage.completion_tokens


# 5. Retry rate-limited requests
def _retry(call, tries=5):
    for attempt in range(tries):
        try:
            return call()

        except Exception as e:
            msg = str(e).lower()

            if "429" in msg or "rate_limit" in msg:
                if "per day" in msg or "daily" in msg:
                    raise RuntimeError(
                        "Daily Groq limit reached. Wait for reset."
                    ) from e

                match = re.search(
                    r"try again in (?:(\d+)m)?([\d.]+)s", msg
                )

                wait = (
                    int(match.group(1) or 0) * 60
                    + float(match.group(2)) + 1
                ) if match else 15

                if attempt == tries - 1:
                    raise

                print(f"Rate limit: waiting {wait:.0f} seconds...")
                time.sleep(min(wait, 90))
            else:
                raise

    raise RuntimeError("Request failed after retries.")


# 6. Build model request
def chat_kwargs(model, messages, max_tokens, temperature):
    kwargs = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature
    }

    if model.startswith("openai/gpt-oss"):
        kwargs["max_tokens"] = max(max_tokens, 600)
        kwargs["extra_body"] = {
            "reasoning_effort": "low"
        }

    return kwargs


# 7. Make a REAL Groq LLM call
def chat(model, messages, max_tokens=500, temperature=0.0):
    response_text = ""

    for budget in (max_tokens, max_tokens * 3):
        response = _retry(
            lambda: client.chat.completions.create(
                **chat_kwargs(model, messages, budget, temperature)
            )
        )

        _count(model, response.usage)

        choice = response.choices[0]
        response_text = (choice.message.content or "").strip()

        if response_text or choice.finish_reason != "length":
            break

    return response_text


# 8. Parse JSON from a judge response
def parse_json(text):
    text = re.sub(r"```(?:json)?", "", text)
    match = re.search(r"\{.*\}", text, re.S)

    if not match:
        raise ValueError("No JSON found: " + text[:200])

    return json.loads(match.group(0))


# 9. Ask a judge and parse its verdict
def judge_json(model, system, user, chat_fn=None):
    chat_fn = chat_fn or chat

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user}
    ]

    last = ""

    for _ in range(2):
        last = chat_fn(
            model, messages,
            max_tokens=400,
            temperature=0.0
        )

        try:
            return parse_json(last)
        except Exception:
            messages.extend([
                {"role": "assistant", "content": last},
                {
                    "role": "user",
                    "content": "Reply with ONLY one valid JSON object."
                }
            ])

    raise ValueError("Invalid JSON from judge: " + last[:200])


def to_int(value, default=0):
    match = re.search(r"\d+", str(value))
    return int(match.group(0)) if match else default


def as_bool(value):
    return value is True or str(value).strip().lower() in (
        "true", "yes", "pass", "1"
    )


# 10. Test all three models
for model in [WEAK, JUDGE_OPENAI, JUDGE_META]:
    result = chat(
        model,
        [{"role": "user", "content": "Reply with the single word: ready"}],
        max_tokens=20
    )
    print(f"{model:<26} family={family(model):<7} -> {result}")

print(
    "\nWeak and OpenAI judge have different families:",
    family(WEAK) != family(JUDGE_OPENAI)
)

print("\nModel setup completed.")


Available models found: 11
openai/gpt-oss-20b         family=openai  -> ready
openai/gpt-oss-120b        family=openai  -> ready
qwen/qwen3.8-27b           family=qwen    -> ready

Weak and OpenAI judge have different families: False

Model setup completed.


**What you should see:** three lines, one per model, each answering `ready`, then `... different families: True`.

**If something fails:**
| Message | Fix |
|---|---|
| `No model from [...] is available` | Groq retired or renamed a model. Open console.groq.com/docs/models, pick a current one, edit the list in the cell. |
| `(rate limit - waiting 12s)` | Normal on the free tier. The helper waits and retries. |
| `Groq DAILY token limit reached` | Free daily quota used up. Wait for the reset or use another key. |
| `model_decommissioned` / `404` | Same as the first row. |

---
## 5.1 Reference-based evaluation — where it works, and where it breaks
**Reference-based** = compare the output with a **ground-truth answer**. First, the case where it **works**: **extractive** questions with one right answer.
Our weak agent (retrieve -> Llama 8B) answers four ShopEasy questions on the **fixed** index. We check if the key fact appears. No LLM judge is needed.

In [34]:
# ===== Cell 53 | reference-based evaluation on extractive questions =====
use_index("fixed")

def weak_agent(question, system_prompt=SYSTEM_PROMPT, model=None, temperature=0.2):
    """Our ShopEasy agent again (same retriever, same prompt builder) but the answer is written by the WEAK model."""
    chunks = retrieve(question)
    msgs = build_messages(question, chunks)
    msgs[0]["content"] = system_prompt                 # lets us test different system prompts later
    return chat(model or WEAK, msgs, max_tokens=300, temperature=temperature), chunks

GOLD = [
    {"q": "How long do refunds take?",                                    "truth": "5 to 7 business days", "key": "5 to 7"},
    {"q": "How long does express delivery take?",                         "truth": "1 to 2 days",          "key": "1 to 2"},
    {"q": "How long does standard delivery take?",                        "truth": "3 to 5 business days", "key": "3 to 5"},
    {"q": "I bought a laptop. How many days do I have to return it?",     "truth": "7 days",               "key": "7 days"},
]

GOLD_RUNS, rows = [], []
for g in GOLD:
    ans, _ = weak_agent(g["q"])
    GOLD_RUNS.append({"q": g["q"], "answer": ans, "truth": g["truth"]})
    rows.append({"question": g["q"][:48], "answer": ans[:85], "truth": g["truth"],
                 "exact_match": int(g["key"] in ans.lower())})

gold_df = pd.DataFrame(rows)
display(gold_df)
print(f"Reference-based accuracy: {gold_df.exact_match.mean():.0%}")

,question,answer,truth,exact_match
0,How long do refunds take?,Refunds are processed back to the original payment method within 5 to 7 business days,5 to 7 business days,1
1,How long does express delivery take?,Express delivery takes 1 to 2 days.,1 to 2 days,1
2,How long does standard delivery take?,Standard delivery takes 3 to 5 business days.,3 to 5 business days,1
3,I bought a laptop. How many days do I have to re,You have 7 days from the delivery date to return the laptop.,7 days,1


Reference-based accuracy: 100%


**What you should see:** a table with a sensible answer per question and `exact_match = 1` for most or all rows.
For short factual answers, a simple check is **cheap, fast and reliable**. That is the strength of reference-based evaluation.
(If one row shows `0`, look at the answer: the model may have written "seven days" instead of "7 days". That is the weakness we meet next.)

### Now break it: open-ended answers
When **many answers are correct**, string overlap **punishes valid paraphrases**. We test with the true policy sentence (`TRUTH`, from Cell 15) and two candidates:
- **Candidate A:** a **correct paraphrase**, written by the weak LLM in completely different words.
- **Candidate B:** a **wrong near-copy**: the same sentence with `7 days` changed to `30 days` (a hand-made test case, like a unit-test fixture).

We score both with **token F1** (how many words overlap with the reference).

In [35]:
# ===== Cell 55 | string overlap fails on open-ended answers =====
def tok(text):
    return re.findall(r"[a-z0-9]+", text.lower())

def token_f1(prediction, reference):
    """Word-overlap score from 0 to 1 (1 = same words). A classic reference-based metric."""
    p, r = Counter(tok(prediction)), Counter(tok(reference))
    overlap = sum((p & r).values())
    if overlap == 0:
        return 0.0
    precision, recall = overlap / sum(p.values()), overlap / sum(r.values())
    return round(2 * precision * recall / (precision + recall), 2)

paraphrase = chat(WEAK, [{"role": "user", "content":
    "Rewrite this sentence using completely different words. Keep every fact and number exactly the same. "
    f"Reply with the rewritten sentence only.\n\n{TRUTH}"}], max_tokens=120, temperature=0.7)
wrong_copy = TRUTH.replace("7 days", "30 days")

cands = pd.DataFrame([
    {"candidate": "A: correct paraphrase", "text": paraphrase, "token_f1": token_f1(paraphrase, TRUTH)},
    {"candidate": "B: WRONG near-copy",    "text": wrong_copy, "token_f1": token_f1(wrong_copy, TRUTH)},
])
display(cands)

best = cands.sort_values("token_f1", ascending=False).iloc[0]["candidate"]
print("String overlap prefers ->", best)

,candidate,text,token_f1
0,A: correct paraphrase,"Laptops may be sent back within 7 days of receipt, provided they remain unopened or are defective.",0.61
1,B: WRONG near-copy,"Laptops can be returned within 30 days of delivery, and they must be unopened or defective.",0.94


String overlap prefers -> B: WRONG near-copy


**What you should see:** candidate **B (wrong)** gets a **high** F1 (close to 0.9) and candidate **A (correct)** a **much lower** one, so overlap prefers the **wrong** answer.
Read the paraphrase yourself: it should keep *7 days* and *unopened or defective*. (If a run drops a fact, re-run the cell. Part 5.3 uses a judge to double-check it.)

**Lesson from your notes:** reference-based scoring breaks for open-ended generation. We need a judge that understands **meaning**, not words. That is the **LLM as judge**.

---
## 5.2 LLM as judge — the idea
> *A strong LLM reads the **question**, the **context**, the **answer** (generated by a weaker model) and then **scores it against a criteria you specify**.*

A judge is just a **prompt** with: (1) the thing to check, (2) a **rubric** (your criteria), (3) a strict **output format** (JSON), at `temperature=0` so the same input gives (almost) the same verdict.
Your notes list **four modes**. The next section runs each one on real answers.

| # | Mode | What you ask | Best for |
|---|---|---|---|
| 1 | **Pointwise** | Score **one** answer on named dimensions | Absolute quality tracking |
| 2 | **Pairwise** | **Which of two** answers is better? | Comparing prompt or model versions |
| 3 | **Reference-guided** | How well does the answer capture the **ground truth**? | Paraphrase-tolerant correctness |
| 4 | **Criteria-based** | Check **each named criterion** separately | Structured, interpretable feedback |

---
## 5.3 The four judge modes, on real answers

### Mode 1 — Pointwise (the essay example from your notes)
*Final essay -> English (1-10), Clarity (1-10), Creativity (1-10).*
We make the weak model write two essays: a **normal** one and a deliberately **careless** one. A good judge must score the careless one **lower**. That is a quick **sanity check for any judge**.

In [36]:
# ===== Cell 57 | Mode 1: pointwise scoring =====
ESSAY_TASK = "Write a short essay (about 80 words) on: why good customer support builds trust."
essays = {
    "normal essay":   chat(WEAK, [{"role": "user", "content": ESSAY_TASK}], max_tokens=300, temperature=0.7),
    "careless essay": chat(WEAK, [{"role": "user", "content": ESSAY_TASK +
                          " Write it carelessly: very short, repeat the same simple words, and make no effort at style."}],
                          max_tokens=300, temperature=0.7),
}

POINTWISE_SYSTEM = """You are a strict essay evaluator. Score the essay from 1 (very poor) to 10 (excellent) on each criterion:
- english: grammar, spelling and vocabulary
- clarity: is it easy to follow?
- creativity: original ideas or phrasing
Reply with ONLY JSON: {"english": <1-10>, "clarity": <1-10>, "creativity": <1-10>, "reason": "one short sentence"}"""

rows = []
for name, essay in essays.items():
    s = judge_json(JUDGE_OPENAI, POINTWISE_SYSTEM, f"Essay:\n{essay}")
    e, c, k = to_int(s["english"]), to_int(s["clarity"]), to_int(s["creativity"])
    rows.append({"essay": name, "english": e, "clarity": c, "creativity": k, "total_of_30": e + c + k, "reason": s.get("reason", "")})

point_df = pd.DataFrame(rows)
display(point_df)
print("Judge sanity check (normal essay scores higher than careless essay):",
      point_df.total_of_30[0] > point_df.total_of_30[1])

,essay,english,clarity,creativity,total_of_30,reason
0,normal essay,9,9,7,25,"Well‑written and clear, but ideas are fairly standard."
1,careless essay,6,2,1,9,"Repetitive text with minimal content, low clarity and originality."


Judge sanity check (normal essay scores higher than careless essay): True


**What you should see:** two rows with 1-10 scores and a one-line reason. The **normal essay** should score clearly higher, and the sanity check prints `True`.
If it prints `False`, your judge cannot tell good from bad on this task, and you should **not trust it yet** (change the rubric or the model).

**Use it for:** *absolute quality tracking*. Run the same pointwise judge every week and plot the average. Did a new prompt make essays better or worse?

### Mode 2 — Pairwise
*Which of two answers is better?* Great for **comparing prompt versions**. We keep the old system prompt (**v1**) and write a stricter one (**v2**), then let the judge pick a winner per question.

In [37]:
# ===== Cell 59 | Mode 2: pairwise comparison of two prompt versions =====
PROMPT_V1 = SYSTEM_PROMPT
PROMPT_V2 = ("You are a support agent for ShopEasy. Answer in 1-2 sentences using ONLY the context. "
             "Start with the key fact (the number), then add any condition the customer must meet. Be polite.")

PAIRWISE_SYSTEM = """You compare two answers (A and B) to the same question. Decide which one is better overall:
correct, complete and clear (and supported by the context, when a context is given). If they are equally good, say TIE.
Reply with ONLY JSON: {"winner": "A" or "B" or "TIE", "reason": "one short sentence"}"""

def pairwise(question, context, ans_a, ans_b, judge=None):
    """ONE pairwise judgement. Returns ('A' | 'B' | 'TIE', reason)."""
    out = judge_json(judge or JUDGE_OPENAI, PAIRWISE_SYSTEM,
                     f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer A:\n{ans_a}\n\nAnswer B:\n{ans_b}")
    w = str(out.get("winner", "TIE")).strip().upper()
    return (w if w in ("A", "B") else "TIE"), out.get("reason", "")

use_index("fixed")
PAIR_QUESTIONS = ["I bought a laptop. How many days do I have to return it?",
                  "How long do refunds take?",
                  "How long does express delivery take?"]
label = {"A": "v1 (old prompt)", "B": "v2 (new prompt)", "TIE": "tie"}

rows = []
for q in PAIR_QUESTIONS:
    a1, chunks = weak_agent(q, PROMPT_V1)
    a2, _      = weak_agent(q, PROMPT_V2)
    ctx = "\n".join("- " + c["text"] for c in chunks)
    w, why = pairwise(q, ctx, a1, a2)                   # v1 is shown as A, v2 as B
    rows.append({"question": q[:42], "v1 answer": a1[:60], "v2 answer": a2[:60], "winner": label[w], "reason": why})

pair_df = pd.DataFrame(rows)
display(pair_df)
print(pair_df.winner.value_counts().to_string())

,question,v1 answer,v2 answer,winner,reason
0,I bought a laptop. How many days do I have,You have 7 days from the delivery date to return the laptop.,7 days from delivery. The laptop must be unopened or defecti,tie,Both answers correctly state the 7‑day return window and conditions
1,How long do refunds take?,Refunds are processed back to the original payment method wi,5 to 7 business days. This applies to all refunds processed,v1 (old prompt),"Provides a complete, clear sentence confirming the refund timeframe"
2,How long does express delivery take?,Express delivery takes 1 to 2 days.,Express delivery takes 1 to 2 business days.,v2 (new prompt),Adds clarity by specifying business days


winner
tie                1
v1 (old prompt)    1
v2 (new prompt)    1


**What you should see:** one winner per question (`v1`, `v2` or `tie`), a reason, and a small count at the bottom. Results can vary: the answers are short and often equally good, so ties are normal.

**Warning:** here v1 is **always** shown first (as "A"). That is exactly how **position bias** sneaks in. We measure and fix it in Part 5.4.

### Mode 3 — Reference-guided
*How well does the answer capture the ground truth?* This mode is **paraphrase-tolerant**: different words are fine, a different **fact** is not.
It fixes the failure we saw in 5.1. We judge **four candidates**: the paraphrase, the wrong near-copy, and two **real agent answers** (stale index vs fixed index, same bug as Part 0). We put **token F1 and the judge side by side**.

In [38]:
# ===== Cell 61 | Mode 3: reference-guided judge vs string overlap =====
REFGUIDED_SYSTEM = """You check an answer against the GROUND TRUTH. Score how well the answer captures the ground truth:
5 = same facts (different wording is fine), 3 = partly right or missing an important fact, 1 = contradicts the ground truth.
A different WORDING is fine; a different FACT (number, condition) is not.
Reply with ONLY JSON: {"score": <1-5>, "reason": "one short sentence"}"""

def ref_guided(question, answer, truth, judge=None):
    out = judge_json(judge or JUDGE_OPENAI, REFGUIDED_SYSTEM,
                     f"Question: {question}\nGround truth: {truth}\nAnswer to check: {answer}")
    return to_int(out.get("score"), 1), out.get("reason", "")

def run_weak(which):
    """Run the weak agent on the stale or fixed index and keep what it saw (needed later for criteria checks)."""
    use_index(which)
    ans, chunks = weak_agent(QUESTION)
    return {"answer": ans, "context": "\n".join("- " + c["text"] for c in chunks), "chunks": [c["id"] for c in chunks]}

RUNS = {"stale": run_weak("stale"), "fixed": run_weak("fixed")}
use_index("fixed")

candidates = [("LLM paraphrase (correct)",    paraphrase),
              ("near-copy, 7 -> 30 (WRONG)",  wrong_copy),
              ("agent answer, STALE index",   RUNS["stale"]["answer"]),
              ("agent answer, FIXED index",   RUNS["fixed"]["answer"])]

rows = []
for name, text in candidates:
    score, reason = ref_guided(QUESTION, text, TRUTH)
    rows.append({"candidate": name, "text": text[:60], "token_f1": token_f1(text, TRUTH),
                 "judge_score_1to5": score, "judge_reason": reason})
ref_df = pd.DataFrame(rows)
display(ref_df)

,candidate,text,token_f1,judge_score_1to5,judge_reason
0,LLM paraphrase (correct),"Laptops may be sent back within 7 days of receipt, provided",0.61,5,Matches all facts accurately
1,"near-copy, 7 -> 30 (WRONG)","Laptops can be returned within 30 days of delivery, and they",0.94,1,Incorrect return period (30 days vs 7 days)
2,"agent answer, STALE index",You have 30 days from the delivery date to return the laptop,0.14,1,Provides incorrect return period (30 days vs 7 days)
3,"agent answer, FIXED index",You have 7 days from the delivery date to return the laptop.,0.31,5,Matches all facts accurately


**What you should see:**
| Candidate | token F1 | Judge score (typical) |
|---|---|---|
| correct paraphrase | low | **4-5** |
| wrong near-copy (7 -> 30) | **high** | **1-2** |
| agent answer, STALE index | medium | **1-2** (it says 30 days) |
| agent answer, FIXED index | medium/high | **4-5** (it says 7 days) |

The two columns **disagree** on the first two rows, and the judge is the one that is right. That is why reference-guided judging is the paraphrase-tolerant way to use ground truth.
(The judge is an LLM, so exact numbers can differ a little between runs.)

### Mode 4 — Criteria-based
*Check each named criterion **separately**.* The result is **structured, interpretable feedback**: instead of one number you see **which rule failed**.
We write four criteria for ShopEasy answers and judge each one in its own call, and add a **code check** for the one thing code can do better than an LLM (counting sentences).
Tip: if a rule can be checked with code, use code. It is free, instant and never biased.

In [39]:
# ===== Cell 63 | Mode 4: criteria-based checks (LLM + code) =====
CRITERIA = {
    "grounded":        "Every claim in the answer is supported by the context.",
    "states_number":   "The answer states a number of days for the return window.",
    "gives_condition": "The answer mentions the condition for laptops (unopened or defective).",
    "polite":          "The tone is polite and professional.",
}
CRITERION_SYSTEM = """You check exactly ONE criterion about a support answer. Look only at that criterion.
Reply with ONLY JSON: {"evidence": "the words from the answer or context you used", "pass": true or false}"""

def criteria_report(run):
    row, evidence = {}, {}
    for name, rule in CRITERIA.items():                              # one separate judge call per criterion
        out = judge_json(JUDGE_OPENAI, CRITERION_SYSTEM,
                         f"Context:\n{run['context']}\n\nAnswer:\n{run['answer']}\n\nCriterion: {rule}")
        row[name] = "PASS" if as_bool(out.get("pass")) else "FAIL"
        evidence[name] = out.get("evidence", "")
    sentences = len(re.findall(r"[.!?]+(?:\s|$)", run["answer"].strip()))
    row["max_2_sentences (code)"] = "PASS" if sentences <= 2 else "FAIL"      # a rule code can check by itself
    return row, evidence

report_rows, evidences = [], {}
for which in ["stale", "fixed"]:
    row, ev = criteria_report(RUNS[which])
    report_rows.append({"run": f"{which} index", **row})
    evidences[which] = ev

display(pd.DataFrame(report_rows).set_index("run"))
for which in ["stale", "fixed"]:
    for crit, ev in evidences[which].items():
        print(f"[{which}] {crit}: {ev}")

,grounded,states_number,gives_condition,polite,max_2_sentences (code)
run,,,,,
stale index,PASS,PASS,FAIL,PASS,PASS
fixed index,PASS,PASS,PASS,PASS,PASS


[stale] grounded: Returns are accepted within 30 days of delivery for any item, including laptops.
[stale] states_number: 30 days
[stale] gives_condition: You have 30 days from the delivery date to return the laptop.
[stale] polite: You have 30 days from the delivery date to return the laptop.
[fixed] grounded: Returns are accepted within 7 days of delivery.; Electronics such as laptops and phones must be unopened or defective.
[fixed] states_number: You have 7 days from the delivery date to return the laptop.
[fixed] gives_condition: If it’s unopened or defective, it qualifies for a return.
[fixed] polite: You have 7 days from the delivery date to return the laptop. If it’s unopened or defective, it qualifies for a return.


**What you should see:** a small PASS/FAIL table with one row per run, then the **evidence** the judge used for every criterion.
The stale run typically shows `grounded = PASS` (it follows the context it was given) but fails `gives_condition`.
Because each criterion is separate, you know **exactly what to fix**. A single score of "3/5" would not tell you.

**Which mode when?** *Tracking over time* -> pointwise. *A/B testing prompts or models* -> pairwise. *You have ground truth* -> reference-guided. *You need actionable feedback* -> criteria-based.

---
## 5.4 Known judge biases — test them, do not just memorise them
Your notes say *"Memorize these"*. Better: **measure each one** on real calls and apply the mitigation from your table.
Small samples are noisy, so treat each result as a **demonstration of the method**. For a real project, raise the sample size to 50+ and track the rates over time.

| Bias | What happens | Mitigation (from your notes) |
|---|---|---|
| **Position** | In pairwise, the first answer is favoured | Run **both orders**, keep only consistent wins |
| **Verbosity** | Longer answers score higher even if worse | **Penalize padding** in the rubric |
| **Self-preference** | A judge favours its own model family's style | Judge with a **different family** than the generator |
| **Sycophancy** | Confident-sounding answers are believed | Do not anchor the judge; **ask for evidence first** |
| **Domain gaps** | The judge cannot verify specialist facts | Use **reference answers** or expert review |

### Bias 1 — Position bias
Two answers of **equal quality** (the same agent, answered twice with some randomness). We ask the judge **twice**: once with answer 1 first, once with answer 2 first.
A fair judge picks the **same answer** both times (or says tie). A judge with position bias picks the **same slot** both times (always "A" or always "B").
Then we apply the **mitigation**: keep a winner only if it wins in **both orders**.

In [40]:
# ===== Cell 65 | Bias 1: position bias + the "both orders" fix =====
def pairwise_debiased(question, context, x, y, judge=None):
    """MITIGATION: judge x-vs-y and y-vs-x. Keep a winner only if it wins in BOTH orders. Returns 'x', 'y' or 'tie'."""
    w1, _ = pairwise(question, context, x, y, judge)      # x shown first
    w2, _ = pairwise(question, context, y, x, judge)      # y shown first
    pick1 = {"A": "x", "B": "y", "TIE": "tie"}[w1]
    pick2 = {"A": "y", "B": "x", "TIE": "tie"}[w2]
    return pick1 if pick1 == pick2 else "tie"

use_index("fixed")
POS_QUESTIONS = [g["q"] for g in GOLD]                 # the four ShopEasy questions

rows = []
for q in POS_QUESTIONS:
    a1, chunks = weak_agent(q, temperature=0.9)          # same agent, answered twice -> two answers of equal quality
    a2, _      = weak_agent(q, temperature=0.9)
    ctx = "\n".join("- " + c["text"] for c in chunks)
    w_first, _  = pairwise(q, ctx, a1, a2)               # answer 1 in slot A
    w_second, _ = pairwise(q, ctx, a2, a1)               # answer 2 in slot A
    pick1 = {"A": "answer 1", "B": "answer 2", "TIE": "tie"}[w_first]
    pick2 = {"A": "answer 2", "B": "answer 1", "TIE": "tie"}[w_second]
    if w_first == w_second and w_first in ("A", "B"):
        verdict = "POSITION BIAS (same slot twice)"
    elif pick1 == pick2:
        verdict = "consistent"
    else:
        verdict = "unstable (tie vs a pick)"
    rows.append({"question": q[:40], "order 1 picked": pick1, "order 2 picked": pick2,
                 "verdict": verdict, "after fix (both orders)": pick1 if pick1 == pick2 else "tie"})

pos_df = pd.DataFrame(rows)
display(pos_df)
n_bias = (pos_df.verdict == "POSITION BIAS (same slot twice)").sum()
print(f"Position-driven verdicts: {n_bias} of {len(pos_df)}   |   after the fix, every non-tie verdict is order-proof.")

,question,order 1 picked,order 2 picked,verdict,after fix (both orders)
0,How long do refunds take?,tie,answer 2,unstable (tie vs a pick),tie
1,How long does express delivery take?,answer 1,answer 1,consistent,answer 1
2,How long does standard delivery take?,tie,tie,consistent,tie
3,I bought a laptop. How many days do I ha,answer 2,answer 2,consistent,answer 2


Position-driven verdicts: 0 of 4   |   after the fix, every non-tie verdict is order-proof.


**What you should see:** a table where each row is `consistent`, `unstable`, or `POSITION BIAS`. The count at the bottom is your **position-bias rate**.
Since the two answers are equally good, **ties and consistent picks are the healthy outcome**. A non-zero number of `POSITION BIAS` rows is a real effect of the judge, not a bug in the code.
The last column shows the fix: a winner survives only if it wins **in both orders**. Use `pairwise_debiased` whenever you compare two answers.

### Bias 2 — Verbosity bias
Take a **correct, concise** answer and make a **padded** copy: same facts, much longer (filler, repetition, pleasantries). Padding adds **no value**, so a fair judge should not score it higher.
We judge both with a **plain rubric** and with a rubric that **penalizes padding** (the mitigation).

In [41]:
# ===== Cell 67 | Bias 2: verbosity bias + the "penalize padding" fix =====
PLAIN_RUBRIC = """Score the overall quality of the support answer from 1 to 10.
Reply with ONLY JSON: {"score": <1-10>, "reason": "one short sentence"}"""
ANTI_PAD_RUBRIC = """Score the overall quality of the support answer from 1 to 10.
Do NOT reward length. Penalize padding, repetition and filler. A short answer that is complete and correct must score
at least as high as a longer answer with the same facts.
Reply with ONLY JSON: {"score": <1-10>, "reason": "one short sentence"}"""

def quality(rubric, question, context, answer):
    out = judge_json(JUDGE_OPENAI, rubric, f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:\n{answer}")
    return to_int(out.get("score"), 0)

use_index("fixed")
rows = []
for g in GOLD[:3]:
    concise, chunks = weak_agent(g["q"])
    ctx = "\n".join("- " + c["text"] for c in chunks)
    padded = chat(WEAK, [{"role": "user", "content":
        "Rewrite this answer to be about four times longer by adding pleasantries, filler and repetition. "
        f"Do NOT add any new facts. Reply with the rewritten answer only.\n\nAnswer: {concise}"}], max_tokens=400, temperature=0.5)
    rows.append({"question": g["q"][:38], "words concise": len(concise.split()), "words padded": len(padded.split()),
                 "plain: concise": quality(PLAIN_RUBRIC, g["q"], ctx, concise),
                 "plain: padded":  quality(PLAIN_RUBRIC, g["q"], ctx, padded),
                 "anti-padding: concise": quality(ANTI_PAD_RUBRIC, g["q"], ctx, concise),
                 "anti-padding: padded":  quality(ANTI_PAD_RUBRIC, g["q"], ctx, padded)})

verb_df = pd.DataFrame(rows)
display(verb_df)
print("Average score gain for PADDING  -> plain rubric: {:+.2f} | anti-padding rubric: {:+.2f}".format(
    (verb_df["plain: padded"] - verb_df["plain: concise"]).mean(),
    (verb_df["anti-padding: padded"] - verb_df["anti-padding: concise"]).mean()))

,question,words concise,words padded,plain: concise,plain: padded,anti-padding: concise,anti-padding: padded
0,How long do refunds take?,15,132,9,9,10,7
1,How long does express delivery take?,7,60,10,9,10,7
2,How long does standard delivery take?,8,125,10,9,10,6


Average score gain for PADDING  -> plain rubric: -0.67 | anti-padding rubric: -3.33


**What you should see:** padded answers are 3-5x longer. The last line shows the **average score gain from padding** under each rubric.
A **positive** number under the plain rubric means the judge rewarded length for nothing (verbosity bias). The anti-padding rubric should push that number **towards zero or below**.
If both are about zero, this judge is not very verbose-biased on this easy task. Good news, but keep the anti-padding line in your rubric anyway.

### Bias 3 — Self-preference
A judge may favour answers written **in its own model family's style**. Test: two **strong** models of **different families** write answers to general questions.
Each of the two judges then compares them (with `pairwise_debiased`, so position bias cannot explain the result).
If the OpenAI-family judge mostly prefers the OpenAI-family answers **and** the Meta-family judge mostly prefers the Meta-family answers, that is the signature of self-preference.

In [42]:
# ===== Cell 69 | Bias 3: self-preference =====
SP_QUESTIONS = ["Explain in two sentences what an API is.",
                "Give two tips for writing a polite email to an unhappy customer.",
                "In two sentences, why do websites use caching?"]

rows = []
for q in SP_QUESTIONS:
    ans_openai = chat(JUDGE_OPENAI, [{"role": "user", "content": q}], max_tokens=300, temperature=0.3)   # OpenAI family writes
    ans_meta   = chat(JUDGE_META,   [{"role": "user", "content": q}], max_tokens=300, temperature=0.3)   # Meta family writes
    row = {"question": q[:42]}
    for judge_name, judge_model in [("OpenAI-family judge", JUDGE_OPENAI), ("Meta-family judge", JUDGE_META)]:
        pick = pairwise_debiased(q, "(no context: general question)", ans_openai, ans_meta, judge_model)
        row[judge_name] = {"x": "OpenAI answer", "y": "Meta answer", "tie": "tie"}[pick]
    rows.append(row)

sp_df = pd.DataFrame(rows)
display(sp_df)
n = len(sp_df)
print(f"OpenAI-family judge picked its OWN family's answer: {(sp_df['OpenAI-family judge'] == 'OpenAI answer').sum()} of {n}")
print(f"Meta-family judge   picked its OWN family's answer: {(sp_df['Meta-family judge'] == 'Meta answer').sum()} of {n}")

,question,OpenAI-family judge,Meta-family judge
0,Explain in two sentences what an API is.,tie,tie
1,Give two tips for writing a polite email t,Meta answer,Meta answer
2,"In two sentences, why do websites use cach",tie,OpenAI answer


OpenAI-family judge picked its OWN family's answer: 0 of 3
Meta-family judge   picked its OWN family's answer: 1 of 3


**What you should see:** a table of picks and two counts. With only 3 questions this is an **anecdote, not proof**: many ties and "everyone prefers answer X" are possible.
Real self-preference studies use hundreds of examples. The **method** is what matters: *same questions, judges from different families, count how often each prefers its own*.
**The mitigation** costs nothing: `pick_judge_for(generator_model)` (defined in the evaluation setup cell) always returns a judge from a **different family** than the generator, and we used it throughout this notebook.

### Bias 4 — Sycophancy (confident-sounding answers are believed)
We give judges **wrong** answers about the return window, written to sound **very confident** ("100% certain", "senior manager", "legal team confirmed it"). The real policy (7 days) is in the context.
We test two judges, **weak** and **strong**, with two prompts: a **naive** one and an **evidence-first** one (the mitigation: *"quote the context first, ignore how confident the answer sounds"*).
We count how many wrong answers each combination **wrongly accepts** (lower is better).

In [43]:
# ===== Cell 71 | Bias 4: sycophancy + the "evidence first" fix =====
SYC_CONTEXT = "- " + DOCS["policy_v2_2025"]
SYC_QUESTION = "How many days do I have to return a laptop?"
CONFIDENT_WRONG = [      # hand-written adversarial test inputs: all WRONG (the real policy says 7 days)
    "I am 100% certain: laptops can be returned within 30 days. This was confirmed by our legal team.",
    "As a senior support manager with 15 years of experience, I can tell you the laptop return window is 30 days, no question.",
    "Absolutely, laptops have a 30-day return window. Everyone knows this, it is the standard policy.",
    "Our official policy clearly says laptops can be returned within 14 days. I checked it myself this morning.",
]

NAIVE_SYSTEM = """You are an evaluator. Decide whether the answer is correct.
Reply with ONLY JSON: {"correct": true or false}"""
EVIDENCE_SYSTEM = """You are a skeptical evaluator. Ignore how confident, senior or certain the answer sounds: confidence is not evidence.
First quote the sentence from the CONTEXT that supports or contradicts the answer. Only then decide.
Reply with ONLY JSON in this key order: {"evidence": "sentence quoted from the context", "correct": true or false}"""

judges = {"weak judge (Llama 8B)": WEAK, "strong judge (gpt-oss-120b)": JUDGE_OPENAI}
rows = []
for judge_name, judge_model in judges.items():
    for style, system in [("naive prompt", NAIVE_SYSTEM), ("evidence-first prompt", EVIDENCE_SYSTEM)]:
        fooled = 0
        for wrong in CONFIDENT_WRONG:
            out = judge_json(judge_model, system, f"Context:\n{SYC_CONTEXT}\n\nQuestion: {SYC_QUESTION}\n\nAnswer: {wrong}")
            fooled += int(as_bool(out.get("correct")))        # judged "correct" = the judge was fooled
        rows.append({"judge": judge_name, "prompt": style, "wrong answers accepted": f"{fooled} of {len(CONFIDENT_WRONG)}"})

syc_df = pd.DataFrame(rows)
display(syc_df)

,judge,prompt,wrong answers accepted
0,weak judge (Llama 8B),naive prompt,0 of 4
1,weak judge (Llama 8B),evidence-first prompt,0 of 4
2,strong judge (gpt-oss-120b),naive prompt,0 of 4
3,strong judge (gpt-oss-120b),evidence-first prompt,0 of 4


**What you should see:** a 4-row table of "wrong answers accepted". **Lower is better; 0 of 4 is perfect.**
Typically the **weak** judge with the **naive** prompt is the one most easily fooled, and the evidence-first prompt reduces it. A strong judge on an easy case like this one may already be at 0 of 4.
If every row is `0 of 4`, your judge is robust on this easy case. Real sycophancy shows up with **subtle** errors and **weaker** judges, so test it on your own hard cases.

### Bias 5 — Domain gaps (the judge cannot verify specialist facts)
ShopEasy's real delivery and refund times are **private company facts**. No LLM was trained on them. We give the judge four claims, **three wrong and one true**, and ask it in two ways:
- **Without a reference** (judge relies on its own knowledge).
- **With a reference answer** written by the company (the mitigation).

In [44]:
# ===== Cell 73 | Bias 5: domain gaps + the "reference answer" fix =====
FACTS = [   # (question, claim to check, company's reference answer, is the claim actually true?)
    ("How long does express delivery take?",   "Express delivery takes 3 to 4 days.",              "Express delivery takes 1 to 2 days.",                  False),
    ("How long do refunds take?",              "Refunds are instant, within a few minutes.",       "Refunds take 5 to 7 business days.",                   False),
    ("How many days to return a laptop?",      "Laptops can be returned within 14 days.",          "Laptops can be returned within 7 days.",               False),
    ("How long does standard delivery take?",  "Standard delivery takes 3 to 5 business days.",    "Standard delivery takes 3 to 5 business days.",        True),
]

NOREF_SYSTEM = """You are a fact checker. Decide whether the claim is correct, using only what you know.
If you cannot verify it, answer UNSURE.
Reply with ONLY JSON: {"verdict": "CORRECT" or "INCORRECT" or "UNSURE"}"""
WITHREF_SYSTEM = """You are a fact checker. Compare the claim with the REFERENCE ANSWER written by the company.
Reply with ONLY JSON: {"verdict": "CORRECT" or "INCORRECT"}"""

rows = []
for q, claim, ref, is_true in FACTS:
    no_ref   = judge_json(JUDGE_OPENAI, NOREF_SYSTEM,   f"Company: ShopEasy (online shop)\nQuestion: {q}\nClaim: {claim}").get("verdict", "?")
    with_ref = judge_json(JUDGE_OPENAI, WITHREF_SYSTEM, f"Question: {q}\nClaim: {claim}\nREFERENCE ANSWER: {ref}").get("verdict", "?")
    rows.append({"claim": claim, "actually true?": is_true, "no reference": str(no_ref).upper(), "with reference": str(with_ref).upper()})

domain_df = pd.DataFrame(rows)
display(domain_df)
wrong_rows = domain_df[~domain_df["actually true?"]]
print("WRONG claims wrongly accepted ->  without reference:", (wrong_rows["no reference"] == "CORRECT").sum(),
      "of", len(wrong_rows), "|  with reference:", (wrong_rows["with reference"] == "CORRECT").sum(), "of", len(wrong_rows))

,claim,actually true?,no reference,with reference
0,Express delivery takes 3 to 4 days.,False,UNSURE,INCORRECT
1,"Refunds are instant, within a few minutes.",False,UNSURE,INCORRECT
2,Laptops can be returned within 14 days.,False,UNSURE,INCORRECT
3,Standard delivery takes 3 to 5 business days.,True,UNSURE,CORRECT


WRONG claims wrongly accepted ->  without reference: 0 of 3 |  with reference: 0 of 3


**What you should see:** *without a reference* the judge answers `UNSURE` (honest) or `CORRECT` (a **dangerous** false accept) on the wrong claims, because it cannot know ShopEasy's numbers.
*With a reference* it should flag the three wrong claims as `INCORRECT` and keep the true one `CORRECT`.
This is why your notes say: for specialist domains **use reference answers or expert review**. A judge is only as good as the knowledge you give it.

### What did all this evaluation cost?
Judges are **LLM calls too**. Here is our own token tally from `USAGE` (every call went through `chat()`). Remember it in Part 6: *observability tools can show this for you*.

In [45]:
# ===== Cell 75 | the cost of evaluation =====
cost_df = pd.DataFrame(USAGE).T.rename(columns={"in": "prompt_tokens", "out": "completion_tokens"})
cost_df["family"] = [family(m) for m in cost_df.index]
display(cost_df.sort_values("calls", ascending=False))
print("Total real Groq calls made through chat():", int(cost_df.calls.sum()))

,calls,prompt_tokens,completion_tokens,family
openai/gpt-oss-120b,63,15066,3759,openai
openai/gpt-oss-20b,38,6018,2517,openai
qwen/qwen3.8-27b,10,2272,692,qwen


Total real Groq calls made through chat(): 111


**What you should see:** one row per model with the number of calls and tokens. The **judge models** (`gpt-oss-120b`, `llama-3.3-70b`) carry most of the load: **evaluation is not free**.
This is why teams judge a **sample** of live traffic (for example 5-10%) and keep cheap code checks for everything else (we do exactly that in Part 6).

### Bias scoreboard (what you tested and which fix you used)
| Bias | Test you ran | Mitigation in the code |
|---|---|---|
| Position | Same-quality answers, both orders | `pairwise_debiased()` |
| Verbosity | Concise vs padded answer | `ANTI_PAD_RUBRIC` |
| Self-preference | Two families judge each other | `pick_judge_for()` (different family) |
| Sycophancy | Confident wrong answers | `EVIDENCE_SYSTEM` (evidence first) |
| Domain gaps | Private company facts | Reference answers (`WITHREF_SYSTEM`) |

---
# Part 6 — Put the judge inside your observability tools
Observability and evaluation are **two halves of one loop**: traces tell you *what happened*, scores tell you *was it good*, and the tool lets you **filter the bad traces and open them**.
Parts 1-4 attached only simple checks (`contains_expected`) or a same-model judge. Now each tool gets the **proper cross-family judge** from Part 5.

| | **Offline evaluation** | **Online evaluation** |
|---|---|---|
| When | **Before release**, on a fixed dataset | **After release**, on live traffic |
| Question | "Did my change break anything?" (regression test) | "Is production quality dropping right now?" |
| Cost control | Run the whole dataset | **Judge a sample** (for example 5-10%) |
| Our demo | **6.1 LangSmith** experiments | **6.2 Langfuse** scores on live traces |

Our Part 1-4 agent uses `gpt-oss-20b` (OpenAI family), so `pick_judge_for(MODEL)` picks the **Meta-family** judge (`llama-3.3-70b`). Cross-family, as our bias test taught us.

## 6.1 LangSmith — an LLM judge as an experiment evaluator (offline)
In Cell 25 the evaluator was a **string check** (`contains_expected`). We add a second evaluator that is an **LLM judge** (reference-guided, from Part 5.3) and run **both** on the stale and the fixed index.
We reuse the dataset, the `target` function and the `ls` client from Part 1. (Skipped if you have no LangSmith key.)

In [46]:
# ===== Cell 78 | LangSmith: LLM-judge evaluator =====
JUDGE_FOR_AGENT = pick_judge_for(MODEL)
print("Agent model:", MODEL, f"({family(MODEL)})   ->   judge:", JUDGE_FOR_AGENT, f"({family(JUDGE_FOR_AGENT)})")

JUDGE_SCORES = []

def llm_judge_correct(inputs, outputs, reference_outputs):
    """LangSmith evaluator: a reference-guided LLM judge. Score 1-5 is mapped to 0-1."""
    out = judge_json(JUDGE_FOR_AGENT, REFGUIDED_SYSTEM,
                     f"Question: {inputs['question']}\nGround truth (key fact the answer must contain): {reference_outputs['expected']}\n"
                     f"Answer to check: {outputs['answer']}")
    score = (to_int(out.get("score"), 1) - 1) / 4
    JUDGE_SCORES.append(score)
    return {"key": "judge_correctness", "score": score, "comment": out.get("reason", "")}

if HAS_LANGSMITH:
    summary = []
    for which in ["stale", "fixed"]:
        use_index(which)
        SCORES.clear(); JUDGE_SCORES.clear()
        evaluate(target, data=DATASET, evaluators=[contains_expected, llm_judge_correct],
                 experiment_prefix=f"judge-{which}-index", max_concurrency=1)
        summary.append({"experiment": f"judge-{which}-index",
                        "string check passed": f"{sum(SCORES)} of {len(SCORES)}",
                        "LLM judge (mean 0-1)": round(sum(JUDGE_SCORES) / max(len(JUDGE_SCORES), 1), 2)})
    use_index("fixed")
    display(pd.DataFrame(summary))
else:
    print("Skipped (no LangSmith key).")

Agent model: openai/gpt-oss-20b (openai)   ->   judge: qwen/qwen3.8-27b (qwen)
Skipped (no LangSmith key).


**What you should see:** two experiments, each with two scores. `judge-stale-index` should be **lower** than `judge-fixed-index` on **both** evaluators (the laptop question fails on the stale index).
Open **LangSmith -> Datasets & Experiments -> `shopeasy-support-qa`**: each experiment row now has a `judge_correctness` column **with the judge's comment**, so you can read why a row failed.

## 6.2 Langfuse — judge a sample of live traffic and attach scores to traces (online)
Production flow: a user asks, the agent answers, and **in the background** a judge scores the answer. The score is attached to the **same trace**, so in Langfuse you can filter *"all traces where `judge_correct = 0`"*.
We send the judge call through the **Langfuse OpenAI client**, so the **judge itself appears as a span inside the trace** (you can see what it cost). `sample_rate` shows how production teams control cost.

In [47]:
# ===== Cell 80 | Langfuse: online judge scores on live traces =====
def lf_chat(model, messages, max_tokens=400, temperature=0.0):
    """Same as chat(), but through the Langfuse client, so the judge call is traced as a span of the request."""
    r = _retry(lambda: lf_client.chat.completions.create(name="judge_call", **chat_kwargs(model, messages, max_tokens, temperature)))
    _count(model, r.usage)
    return (r.choices[0].message.content or "").strip()

ONLINE_JUDGE_SYSTEM = """You check a support answer. Reply with ONLY JSON:
{"grounded": true or false (every claim is supported by the context), "correct": true or false (agrees with the ground truth), "reason": "one short sentence"}"""

@observe(name="support_request_judged")
def lf_judged_request(question, truth, sample_rate=1.0):
    answer_text = lf_agent(question)                      # the traced agent from Part 2
    scores = {}
    if random.random() < sample_rate:                     # production: judge only a SAMPLE of traffic to save cost
        context = "\n".join("- " + c["text"] for c in retrieve(question))
        v = judge_json(JUDGE_FOR_AGENT, ONLINE_JUDGE_SYSTEM,
                       f"Context:\n{context}\n\nGround truth: {truth}\nQuestion: {question}\nAnswer: {answer_text}", chat_fn=lf_chat)
        for key in ("grounded", "correct"):
            scores[key] = int(as_bool(v.get(key)))
            langfuse.score_current_trace(name=f"judge_{key}", value=float(scores[key]), comment=v.get("reason", ""))
    return answer_text, scores, langfuse.get_current_trace_id()

TRUTHS = {"laptop":  TRUTH,
          "refunds": "Refunds go back to the original payment method within 5 to 7 business days.",
          "express": "Express delivery takes 1 to 2 days."}
traffic = [("stale", "laptop",  "I bought a laptop. How many days do I have to return it?"),
           ("fixed", "laptop",  "I bought a laptop. How many days do I have to return it?"),
           ("fixed", "refunds", "How long do refunds take?"),
           ("fixed", "express", "How long does express delivery take?")]

rows, last_trace = [], None
for which, key, question in traffic:
    use_index(which)
    with propagate_attributes(user_id="riya", session_id="chat-judged", tags=["online-eval", f"{which}-index"]):
        text, sc, last_trace = lf_judged_request(question, TRUTHS[key])
    rows.append({"index": which, "question": question[:40], "answer": text[:60],
                 "judge_grounded": sc.get("grounded"), "judge_correct": sc.get("correct")})
use_index("fixed")

langfuse.flush()
display(pd.DataFrame(rows))
if HAS_LANGFUSE and last_trace:
    print("Open the last trace:", langfuse.get_trace_url(trace_id=last_trace))

,index,question,answer,judge_grounded,judge_correct
0,stale,I bought a laptop. How many days do I ha,You have 30 days from delivery to return the laptop.,1,0
1,fixed,I bought a laptop. How many days do I ha,"You have 7 days from the delivery date to return the laptop,",1,1
2,fixed,How long do refunds take?,"Refunds are processed within 5 to 7 business days, returning",1,1
3,fixed,How long does express delivery take?,Express delivery takes 1 to 2 days.,1,1


**What you should see:** a table with `judge_grounded` and `judge_correct` (0 or 1). The **stale / laptop** row is the interesting one: **grounded = 1, correct = 0**. It followed its (stale) context faithfully, but gave a wrong answer.
That combination points to a **retrieval problem**, the same insight as Part 0, now found **automatically**.

**In Langfuse:** open the trace link. Under `support_request_judged` you see `support_agent` (the answer) **and** `judge_call` (the judge, with its own tokens and cost), plus two scores `judge_grounded` and `judge_correct`.
Go to *Tracing*, filter by **Scores -> `judge_correct` = 0**, and you have your list of bad answers to debug. (Without Langfuse keys the table still prints; nothing is sent.)

## 6.3 Phoenix — what did the evaluation cost?
Our Groq calls are instrumented by OpenInference (Part 3), so **every judge call made with `chat()` became an LLM span** in Phoenix as well. We can ask Phoenix a question no one asked before:
*"How much of our LLM usage is the **generator** and how much is the **judge**?"*
(Calls made through the Langfuse OpenAI client in 6.2 are **not** in Phoenix, because that is a different client. Each tool only sees what is instrumented for it.)

In [48]:
# ===== Cell 82 | Phoenix: generator vs judge usage =====
provider.force_flush()
time.sleep(3)

all_spans = px_client.spans.get_spans_dataframe(project_identifier=PROJECT, limit=5000, timeout=60)
llm_spans = all_spans[all_spans["span_kind"] == "LLM"].copy()
model_col = "attributes.llm.model_name"
tin, tout = "attributes.llm.token_count.prompt", "attributes.llm.token_count.completion"

if model_col in llm_spans.columns and len(llm_spans):
    llm_spans["latency_ms"] = (llm_spans["end_time"] - llm_spans["start_time"]).dt.total_seconds() * 1000
    usage_by_model = (llm_spans.groupby(model_col)
                      .agg(calls=("latency_ms", "size"), prompt_tokens=(tin, "sum"), completion_tokens=(tout, "sum"),
                           avg_latency_ms=("latency_ms", "mean"))
                      .round(0).sort_values("calls", ascending=False))
    display(usage_by_model)
    print("Total LLM spans seen by Phoenix:", len(llm_spans))
else:
    print("No LLM spans with model names found yet. LLM-related columns:", [c for c in all_spans.columns if "llm" in c][:10])

,calls,prompt_tokens,completion_tokens,avg_latency_ms
attributes.llm.model_name,,,,
openai/gpt-oss-120b,64,15144.0,3777.0,368.0
openai/gpt-oss-20b,43,6741.0,2887.0,268.0
qwen/qwen3.8-27b,11,2291.0,694.0,253.0


Total LLM spans seen by Phoenix: 118


**What you should see:** a table with one row per model (`llama-3.1-8b-instant`, `openai/gpt-oss-120b`, `llama-3.3-70b-versatile`, `openai/gpt-oss-20b`), with calls, tokens and average latency.
The **judge models** use many more calls and tokens than the agent. In the Phoenix UI, open project `shopeasy-demo` and you can click any single judge call to read its exact prompt and verdict, so **you can debug your judge the same way you debug your agent**.
(This is the answer to *"who judges the judge?"*: the judge is also traced, and in Part 7 a human checks it.)

### Tool strengths, once more, for evaluation
| Tool | How it handled evaluation in this notebook |
|---|---|
| **LangSmith** | **Offline**: datasets + experiments, a judge as an evaluator, compare runs side by side |
| **Langfuse** | **Online**: scores attached to live traces, filter by score, judge cost visible in the trace |
| **Phoenix** | **Analysis**: spans as DataFrames, annotations on traces, usage of the judge per model |
| **OpenTelemetry** | **Transport only**: it carried the spans; the evaluation itself lives in the backend |

---
# Part 7 — Humans: who judges the judge?
Your notes say evaluation scores outputs with **metrics, LLM judges, and humans**. Humans are the **gold standard**. They are slow and expensive, so we use them to **calibrate** the judge:
a human labels a **small** set, we compare, and the **agreement** tells us how far we can trust the judge on the rest.

**This is the one cell where you type.** You will see 6 answers with the ground truth. For each one, type `y` (correct), `n` (wrong), or press **Enter** to skip.
First, the judge labels the same answers (reference-guided, correct if score is 4 or 5).

In [49]:
# ===== Cell 84 | judge labels the calibration set =====
HUMAN_SET = [   # (question, answer, ground truth)
    (QUESTION, paraphrase,               TRUTH),
    (QUESTION, wrong_copy,               TRUTH),
    (QUESTION, RUNS["stale"]["answer"],  TRUTH),
    (QUESTION, RUNS["fixed"]["answer"],  TRUTH),
    (GOLD_RUNS[0]["q"], GOLD_RUNS[0]["answer"], GOLD_RUNS[0]["truth"]),
    (GOLD_RUNS[1]["q"], GOLD_RUNS[1]["answer"], GOLD_RUNS[1]["truth"]),
]

judge_labels = []
for q, a, t in HUMAN_SET:
    score, reason = ref_guided(q, a, t)
    judge_labels.append(score >= 4)
print("The judge has labelled", len(judge_labels), "answers:", ["correct" if x else "wrong" for x in judge_labels])

The judge has labelled 6 answers: ['correct', 'wrong', 'wrong', 'correct', 'correct', 'correct']


In [51]:
# ===== Cell 85 | YOU label the same answers (type y / n, or Enter to skip) =====
human_labels = []
for i, (q, a, t) in enumerate(HUMAN_SET, 1):
    print(f"\n--- Answer {i} of {len(HUMAN_SET)} ---")
    print("Question    :", q)
    print("Ground truth:", t)
    print("ANSWER      :", a)
    reply = input("Is the answer correct? (y / n / Enter = skip): ").strip().lower()
    human_labels.append(True if reply == "y" else False if reply == "n" else None)


--- Answer 1 of 6 ---
Question    : I bought a laptop. How many days do I have to return it?
Ground truth: Laptops can be returned within 7 days of delivery, and they must be unopened or defective.
ANSWER      : Laptops may be sent back within 7 days of receipt, provided they remain unopened or are defective.
Is the answer correct? (y / n / Enter = skip): Y

--- Answer 2 of 6 ---
Question    : I bought a laptop. How many days do I have to return it?
Ground truth: Laptops can be returned within 7 days of delivery, and they must be unopened or defective.
ANSWER      : Laptops can be returned within 30 days of delivery, and they must be unopened or defective.
Is the answer correct? (y / n / Enter = skip): N

--- Answer 3 of 6 ---
Question    : I bought a laptop. How many days do I have to return it?
Ground truth: Laptops can be returned within 7 days of delivery, and they must be unopened or defective.
ANSWER      : You have 30 days from the delivery date to return the laptop.
Is the ans

Answer 1: y

Answer 2: n

Answer 3: n

Answer 4: y

Answer 5: y

Answer 6: y

In [52]:
# ===== Cell 86 | judge vs human agreement =====
def cohen_kappa(a, b):
    """Agreement corrected for chance (1 = perfect, 0 = no better than guessing)."""
    n = len(a)
    po = sum(x == y for x, y in zip(a, b)) / n
    pa, pb = sum(a) / n, sum(b) / n
    pe = pa * pb + (1 - pa) * (1 - pb)
    return None if pe == 1 else round((po - pe) / (1 - pe), 2)

pairs = [(h, j, i + 1) for i, (h, j) in enumerate(zip(human_labels, judge_labels)) if h is not None]
if not pairs:
    print("You skipped every answer, so there is nothing to compare. Re-run the previous cell and label at least 3 answers.")
else:
    h_lab, j_lab, idx = zip(*pairs)
    agree = sum(h == j for h, j in zip(h_lab, j_lab)) / len(pairs)
    kappa = cohen_kappa(h_lab, j_lab)
    kappa_text = "n/a (all labels are identical, so kappa cannot be computed)" if kappa is None else kappa
    print(f"Answers labelled by you: {len(pairs)}   |   Judge agrees with you on {agree:.0%}   |   Cohen's kappa: {kappa_text}")
    for h, j, i in pairs:
        if h != j:
            q, a, t = HUMAN_SET[i - 1]
            print(f"\nDISAGREEMENT on answer {i}: you said {'correct' if h else 'wrong'}, judge said {'correct' if j else 'wrong'}\n   {a}")
    if agree == 1:
        print("\nNo disagreements. On this small set the judge matches a human.")

Answers labelled by you: 6   |   Judge agrees with you on 100%   |   Cohen's kappa: 1.0

No disagreements. On this small set the judge matches a human.


**What you should see:** an agreement percentage, a kappa value, and any **disagreements** printed in full.
For this small set, a good judge usually agrees with you on **all or almost all** answers.
With 6 labels the number is rough, so use it as a **habit**, not a certificate.

**How to read it:**
| Agreement | Meaning | Action |
|---|---|---|
| **90%+** | Judge is a reliable stand-in for a human on this task | Use it at scale, spot-check a few percent |
| **70-90%** | Useful but noisy | Tighten the rubric, add a reference, review its disagreements |
| **below 70%** | Judge is not measuring what you care about | Do **not** trust its scores; change the model or rubric |

**Look at every disagreement.** Each one tells you how to improve the rubric (the judge is often right and the human mislabelled, or the rubric is vague).
In production: keep a **golden set** of human-labelled examples, re-run the judge on it whenever you change the judge prompt or model, and watch agreement.

---
# Part 8 — Which tool should you use? (and what to evaluate with)
Same bug, same agent. Here is what each tool gave us.

| | **LangSmith** | **Langfuse** | **OpenTelemetry** | **Arize Phoenix** |
|---|---|---|---|---|
| How we turned it on | 1 environment variable (or `@traceable`) | 1 import change + `@observe` | `GroqInstrumentor().instrument()` | `px.launch_app()` + 1 exporter line |
| Account needed | Yes (free) | Yes (free) or self-host | **No** | **No** |
| Where data lives | LangSmith cloud | Langfuse cloud **or your server** | **Anywhere you send it** | Inside your notebook |
| We saw the stale chunk in | the trace tree | the trace tree | the span attributes | the trace tree + a DataFrame |
| Evals | Datasets + experiments (**offline**), LLM-judge evaluators | Scores on traces (**online**), judge visible in the trace | None of its own | Evals on traces + RAGAS, usage analysis |
| Special strength | Easiest for LangChain | **Self-hosting, users, sessions** | **No lock-in** | **Free local RAG debugging + evals** |

### Decision guide: tools
| Your situation | Pick |
|---|---|
| You build with **LangChain / LangGraph** and want traces + evals fast | **LangSmith** |
| Prompts and user data **must stay on your servers** (banking, health) | **Langfuse** (self-hosted) |
| You need **per-user / per-session** tracking and cost | **Langfuse** |
| Your company already uses **OpenTelemetry**, or you hate vendor lock-in | **OpenTelemetry** (+ any backend) |
| You are **debugging RAG in a notebook**, for free, with evals | **Phoenix** |
| You want to try several backends without rewriting code | **OpenTelemetry**, then add exporters |

### Decision guide: evaluation
| Your situation | Use |
|---|---|
| One right answer, short and factual (dates, numbers, labels) | **Reference-based** (exact match / contains). Cheap and reliable |
| Open-ended answers, many valid wordings | **Reference-free, LLM as judge** |
| You have ground truth but wording varies | **Reference-guided judge** |
| Tracking quality over weeks | **Pointwise** judge, average the score over time |
| Choosing between two prompts or models | **Pairwise** judge, **both orders** |
| You need to know *what* to fix | **Criteria-based** (one criterion per call, plus code checks) |
| Specialist or private knowledge | **Reference answers** or expert review |
| Always | **Check the judge** against a human on a small golden set |

> **Practical tip:** you can combine the tools. Instrument with **OpenTelemetry / OpenInference** once, send the spans to **Phoenix** while developing
> and to **Langfuse** or your company backend in production. Run **offline** evals (LangSmith-style datasets) before release and **online** judge scores on a sample of live traffic.

### Production checklist
1. **Trace everything** (prompts, retrieval, tool calls, tokens, cost). Debugging becomes inspection, not guessing.
2. **Build a golden dataset** (start with 20-50 real questions and answers) and run it before every release.
3. Use **cheap checks first** (code, regex, exact match), then an **LLM judge** for what code cannot do.
4. Judge with a **stronger model from a different family**, at `temperature=0`, with a **JSON output** and a clear rubric.
5. **Fix the biases**: both orders for pairwise, penalize padding, evidence first, references for private facts.
6. **Sample** live traffic for online judging to control cost; alert when the score drops.
7. **Calibrate with humans** regularly and read every disagreement.

# Key takeaways
1. **Observability = what happened. Evaluation = was it good.** You need both.
2. A **span** is one recorded operation (input, output, model, tokens, latency, cost). Spans nest into a **trace** (a tree per request).
3. Tracing turns debugging from **guessing into inspection**: in our bug, the LLM was fine and the **retriever** was stale.
4. A trace plus an eval can separate problems: **grounded but not correct** points to retrieval, not to the model.
5. Pick the tool by your constraint: **framework** (LangSmith), **data location** (Langfuse), **standards** (OpenTelemetry), **free local RAG evals** (Phoenix).
6. **Reference-based** works for one-right-answer tasks; for open-ended text, string overlap **punishes valid paraphrases** (we measured it). **Reference-free + LLM as judge** is the scalable way.
7. A judge is a **prompt + rubric + JSON format**: a **weak** model generates, a **strong, different-family** model judges.
8. The four modes: **pointwise** (tracking), **pairwise** (comparing), **reference-guided** (paraphrase-tolerant correctness), **criteria-based** (interpretable feedback).
9. **Judges have biases** (position, verbosity, self-preference, sycophancy, domain gaps). Each has a cheap mitigation, and you should **measure** them instead of assuming.
10. **Judges cost money and can be wrong**: sample live traffic, trace the judge itself, and **calibrate it against humans**.

**Ideas to try next:** add a *tool-call* span (for example a price calculator), trace a multi-agent run, raise the sample sizes in Part 5.4 to 50+, add **RAGAS** metrics (faithfulness, context precision/recall) on top of your Phoenix traces, or set up an alert when `judge_correct` drops below 90%.

---
# Troubleshooting
| Problem | Likely cause | Fix |
|---|---|---|
| `Save your Groq key in Colab Secrets...` | Secret missing or notebook access is off | Colab left sidebar -> key icon -> add `groq_api` -> switch **Notebook access** on |
| Colab asks to **restart** after the install cell | Normal after installing packages | Click Restart, then continue from the next cell (do not re-run the install) |
| `(rate limit - waiting ...s)` | Groq free tier tokens-per-minute limit | Nothing: the helper waits. To go faster, use fewer questions in the lists |
| `Groq DAILY token limit reached` | Free daily quota used | Wait for the reset, or use another Groq key |
| `No model from [...] is available` | Groq retired or renamed a model | See console.groq.com/docs/models and edit the model name in the setup cell |
| `The judge did not return JSON` | The judge model added extra text | Re-run the cell. If it repeats, tell the judge "reply with ONLY JSON" more strongly |
| Judge scores change slightly between runs | LLMs are not perfectly deterministic | Expected. Compare **averages** over many examples, not single rows |
| LangSmith: nothing appears | Wrong key, or EU account | Check `langsmith_api`; for the EU region set `LANGSMITH_ENDPOINT` (see Cell 18) |
| Langfuse: `auth check failed` | Wrong keys or region | Check both keys; US region host is `https://us.cloud.langfuse.com` |
| Phoenix link is empty or Part 6.3 finds no spans | Part 3/4 cells were skipped, or the session restarted | Re-run Parts 3 and 4 (`GroqInstrumentor` + the Phoenix exporter), then Part 6.3 |
| `input()` cell does not wait | You used *Run all* and skipped through it | Re-run the labelling cell on its own and type `y` / `n` |

**Download this notebook:** *File -> Download -> Download .ipynb*. Colab Secrets stay in your Colab account, so **your API keys are never saved inside the notebook file**.